# **단안 CCTV 상자 치수 추정 — 메인 노트북 (최종 모델 · test 채점 · 런타임 · 시각화)**

> 일반 CCTV 한 대의 영상과 렌즈 focal만으로 컨베이어 위 상자의 **가로 w · 세로 d · 높이 h**를 추정한다. 크기를 **레일 기하(계산) · 점유율(STEP 1) · 형태비(STEP 2)** 로 나눈다.
> 이 노트북은 **최종 모델 하나**로 train · val 재현 → **test 채점(한 번)** → **A100 런타임 측정** → **test 영상 시각화**까지 한다. 단계별 개선 · ablation · baseline 미세조정은 `[최최최종]치수추정_ablation_최종.ipynb`.

```
기하 스케일 = 레일 폭 62.3 cm 기준, bbox 아랫변 높이의 픽셀당 cm                      ← 계산 (학습 없음)
프레임      = 트랙마다 가려짐 적은 순 최대 8장 (bbox ≥ 최대의 50% · 간격 0.2초 이상)
토큰        = 프레임 crop → DA3 인코더 (고정) 7 × 7 ⊕ DINOv3-L (고정) 7 × 7 → (49, 2048)
STEP 1      = 토큰 + 기하 특징 5개 → log B,   log w = log(bbox 가로 × 기하 스케일) + log B
STEP 2      = 토큰 + [log focal, log 픽셀당 cm] → log d/w, log h/w
트랙 결과   = 프레임별 예측(좌우반전 TTA · fold 5개 평균)을 트랙마다 평균 → w, d = w × (d/w), h = w × (h/w)
```

| 구성 요소 | 내용 | 파라미터 |
|---|---|---|
| DA3 인코더 | Depth Anything 3 (DA3METRIC-LARGE) 안의 ViT, crop 448 · patch 14 → 7 × 7 | 304.4 M (고정) |
| DINOv3-L | `timm/vit_large_patch16_dinov3.lvd1689m`, crop 448 · patch 16 → 7 × 7 | 303.1 M (고정) |
| STEP 1 · STEP 2 head | 토큰마다 LayerNorm → Linear(2048 → 64) → GELU → 49 × 64 펼침 + 보조 입력 → MLP(256) | 학습 약 1.8 M |

## 실행 방법
1. **런타임 > 런타임 유형 변경 > A100 GPU** (가능하면 높은 RAM). 런타임 결과는 [R-0]에 기록된 환경 기준이다
2. **런타임 > 모두 실행.** 가중치(ablation [2C-13]) · 토큰 캐시가 Drive에 있으므로 학습 · train/val 추출은 건너뛴다 (test 토큰만 처음 추출, 수 분)
3. [E-1]의 val ②가 **1.952 cm**로 재현될 때만 [T-1]이 test를 채점한다 (기록: `OUT2_DIR/TEST_SCORED.txt`)
4. 시각화는 [V-1]의 `VIS_NUM`(test 영상 번호)만 바꿔서 [V-1] · [V-2]를 다시 실행

## 노트북 흐름
| 장 | 내용 | 셀 |
|---|---|---|
| 0 ~ 2 | 설정 · 공용 함수 · 데이터 · 5-fold | [S-1] ~ [D-1] |
| 3 | 프레임 선택 (가려짐 적은 최대 8장) · crop | [P-0] · [P-1] |
| 4 | 토큰 캐시 — DINOv3-L · DA3 인코더 | [T-0] · [T-0b] |
| 5 | 최종 head · 모든 트랙 예측 | [F-1] · [F-2] |
| 6 | train · val 채점 (재현 확인) | [E-1] |
| 7 | **test 채점 (한 번)** · `result.json` · baseline 비교 | [T-1] · [T-2] |
| 8 | bin packing 입력 | [X-1] |
| 9 | **런타임** — 환경 · 배치 크기 · 박스당 시간 · 실시간 | [R-0] ~ [R-4] |
| 10 | **시각화** — test 영상 위 결과 · 상자별 3D | [V-1] · [V-2] |

## 지표
| 지표 | 계산 |
|---|---|
| **① 가로 w 오차** | 매칭 쌍의 \|예측 w − 정답 w\| 평균 (cm · %) |
| **② 치수 오차 (매칭 쌍)** | 세 변을 큰 순으로 정렬해 같은 순위끼리 평균 절대오차 → 상자 평균 (**주 지표**) |
| **③ ② + 벌점** | 놓친 정답은 예측 (0, 0, 0), 정답 없는 트랙은 정답 (0, 0, 0)으로 보고 같은 계산 (**시스템 전체**) |

- 매칭 = 수작업 매핑(트랙 ↔ 정답), REMOVED는 양쪽에서 제외. train은 영상 단위 5-fold OOF, val · test는 fold 모델 5개 평균. 신뢰구간은 영상 단위 bootstrap 95% (2,000회)

# **0. 설정 · 경로**

In [ ]:
# [S-1] Drive 마운트 + 경로 + 한글 폰트 (폰트 설치가 실패해도 계속 진행)
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, pickle, time, math, random, shutil, subprocess, urllib.request, socket

ROOT        = '/content/drive/MyDrive/4학년 1학기 과목별 정리/소프트웨어융합캡스톤디자인'
PROJECT_DIR = f'{ROOT}/BoxSizing'
DATA_DIR    = f'{ROOT}/Dataset/Final Dataset'
CKPT_DIR    = f'{PROJECT_DIR}/checkpoints'
CACHE_DIR   = f'{PROJECT_DIR}/cache'                     # 트래킹 + 레일 캐시 (이미 있음)
OUT_DIR     = f'{PROJECT_DIR}/outputs/step1'             # fold 분할 · OOF 저장소
OUT2_DIR    = f'{PROJECT_DIR}/outputs/step2'             # 제안 방법 예측 CSV (OVMono3D 노트북이 읽음)
LOCAL_DATA  = '/content/data'
MAPPING_XLSX = {sp: f'{PROJECT_DIR}/{sp}_label_mapping.xlsx' for sp in ('train', 'val', 'test')}
for d in (OUT_DIR, OUT2_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)

RUN_TEST = True                                          # train · val 재현 확인 → test 채점 (한 번) → 런타임 · 시각화 (test 영상)
SPLITS = ['train', 'val'] + (['test'] if RUN_TEST else [])


def video_path(sp, vid):
    p = f'{LOCAL_DATA}/{sp}/{vid}.mp4'
    return p if os.path.exists(p) else f'{DATA_DIR}/{sp}/{vid}.mp4'


# ---- 한글 폰트: Drive 캐시 → GitHub(10초) → apt(60초), 모두 실패해도 셀은 멈추지 않음 ----
import matplotlib.font_manager as fm, matplotlib.pyplot as plt
FONT = f'{PROJECT_DIR}/fonts/NanumGothic.ttf'
if not os.path.exists(FONT):
    os.makedirs(os.path.dirname(FONT), exist_ok=True)
    try:
        socket.setdefaulttimeout(10)
        urllib.request.urlretrieve('https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf', FONT)
    except Exception as e:
        print('GitHub 폰트 다운로드 실패 → apt 시도:', type(e).__name__)
        subprocess.run('timeout 60 apt-get -qq -o Acquire::http::Timeout=10 install -y fonts-nanum > /dev/null 2>&1', shell=True)
        if os.path.exists('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'):
            shutil.copy('/usr/share/fonts/truetype/nanum/NanumGothic.ttf', FONT)
    finally:
        socket.setdefaulttimeout(None)
if os.path.exists(FONT) and os.path.getsize(FONT) > 100_000:
    fm.fontManager.addfont(FONT); plt.rcParams['font.family'] = fm.FontProperties(fname=FONT).get_name()
else:
    print('⚠️ 한글 폰트 없음 → 그래프의 한글이 네모로 보일 수 있음 (결과에는 영향 없음)')
plt.rcParams['axes.unicode_minus'] = False
print('SPLITS =', SPLITS)

Mounted at /content/drive
SPLITS = ['train', 'val', 'test']


In [ ]:
# [S-2] 패키지 (timm은 Colab 기본 포함, 없을 때만 설치) + GPU 확인
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True); import timm
print('timm', timm.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv

timm 1.0.30
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [ ]:
# [S-3] 영상 로컬 복사 (crop을 새로 만들 때만 영상을 읽음, Drive에서 직접 읽으면 느림)
for sp in SPLITS:
    os.makedirs(f'{LOCAL_DATA}/{sp}', exist_ok=True)
    subprocess.run(['rsync', '-a', f'{DATA_DIR}/{sp}/', f'{LOCAL_DATA}/{sp}/'], check=True)
    print(sp, len(glob.glob(f'{LOCAL_DATA}/{sp}/*.mp4')), '개 영상')

train 100 개 영상
val 49 개 영상
test 50 개 영상


# **1. 공용 함수** — 장면 상수 · 레일 기하 · 라벨 / 매핑 / 캐시 로더 · 기하 특징 · 지표

In [ ]:
# [L-1] 장면 상수 · 레일 기하 · 라벨 / 매핑 / 캐시 로더 · (트랙, 정답) 표 · 기하 특징
import numpy as np, pandas as pd, cv2

RAIL_WIDTH_CM, SENSOR_W_MM, FRAME_W, FRAME_H = 62.3, 5.79, 1280, 720
BEST_ALPHA = 1.0                                         # bbox 아랫변(y2)에서 픽셀당 cm를 읽음 (STEP 1 결론)
MAP_COL = 'new_tracked_unique_id_VERSION5'               # 매핑 엑셀의 트랙 번호 열


def rail_scale_at_y_vec(near_y, far_y, near_w, far_w, y):
    t = np.clip((near_y - y) / np.maximum(near_y - far_y, 1e-6), 0.0, 1.15)
    return RAIL_WIDTH_CM / np.maximum(near_w + (far_w - near_w) * t, 1.0)


def vanish_y_of(q):
    ny, fy = q['near_y'], q['fy']; denom = fy - ny
    aL, aR = (q['fl'] - q['nl']) / denom, (q['fr'] - q['nr']) / denom
    y_lines = ny + (q['nr'] - q['nl']) / (aL - aR) if abs(aL - aR) > 1e-9 else None
    y_width = ny - q['near_w'] * (ny - fy) / (q['near_w'] - q['far_w'] + 1e-9)
    return float(y_width) if y_lines is None else float((y_lines + y_width) / 2)


def vanish_xy(q):
    y_v = vanish_y_of(q)
    t = (q['near_y'] - y_v) / max(q['near_y'] - q['fy'], 1e-6)
    mid_n, mid_f = (q['nl'] + q['nr']) / 2, (q['fl'] + q['fr']) / 2
    return mid_n + (mid_f - mid_n) * t, y_v


def load_labels(path):
    with open(path) as f:
        videos = json.load(f)['videos']
    out = {}
    for v in videos:
        objs = sorted(v['objects'], key=lambda o: o['box_id'])
        out[v['video_id']] = {'boxes': [{'w': float(o['size_cm']['w']), 'd': float(o['size_cm']['d']),
                                         'h': float(o['size_cm']['h']), 'box_id': int(o['box_id'])} for o in objs],
                              'focal_mm': float(v['camera']['focal_length_mm'])}
    print(f'📄 {os.path.basename(path)}: 영상 {len(out)}개, 상자 {sum(len(v["boxes"]) for v in out.values())}개')
    return out


def load_mapping(path, col=MAP_COL):
    m = pd.read_excel(path); out = {}
    for vid, g in m.groupby('video_id'):
        rows = []
        for _, r in g.sort_values('box_id_json').iterrows():
            try:
                trk = int(float(r[col]))
            except (TypeError, ValueError):
                trk = None
            rows.append({'box_id': int(r['box_id_json']), 'trk': trk,
                         'removed': str(r.get('note', '')).strip().upper() == 'REMOVED'})
        out[str(vid)] = rows
    n_ok = sum(r['trk'] is not None and not r['removed'] for v in out.values() for r in v)
    print(f'🗂 {os.path.basename(path)}: 영상 {len(out)}개 → 트랙과 연결된 GT {n_ok}개')
    return out


def load_cache(d):
    recs = {}
    for p in sorted(glob.glob(f'{d}/*.pkl')):
        with open(p, 'rb') as f:
            r = pickle.load(f)
        recs[r['video_id']] = r
    return recs


def _rep(t, min_side_px=3):
    """대표 프레임 = bbox 면적 최대 (너무 작으면 None)"""
    bb = t['bbox']; k = int(((bb[:, 2] - bb[:, 0]) * (bb[:, 3] - bb[:, 1])).argmax())
    x1, y1, x2, y2 = bb[k]
    return None if (x2 <= x1 + min_side_px or y2 <= y1 + min_side_px) else k


def _row(split, vid, rec, lab, t, k, g, n_g):
    q = rec['quad']; x1, y1, x2, y2 = t['bbox'][k]
    return {'split': split, 'vid': vid, 'tid': int(t['tid']), 'box_id': -1 if g is None else g['box_id'],
            'mapped': g is not None, 'rep_fi': int(t['fi'][k]), 'step': rec['step'],
            'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'bw': x2 - x1, 'bh': y2 - y1,
            'w_gt': np.nan if g is None else g['w'], 'd_gt': np.nan if g is None else g['d'],
            'h_gt': np.nan if g is None else g['h'], 'n_gt': n_g, 'focal_mm': lab['focal_mm'],
            'near_y': q['near_y'], 'far_y': q['far_y'], 'near_w': q['near_w'], 'far_w': q['far_w']}


def build_tables(recs, labels, split, mapping):
    """반환: (매핑 쌍 표, 모든 트랙 표). REMOVED 상자의 트랙은 모든 트랙 표에서도 제외."""
    pairs, alltrk = [], []
    for vid, rec in recs.items():
        if vid not in labels or rec['quad'] is None:
            continue
        lab, rows = labels[vid], mapping.get(vid, [])
        gt_by = {g['box_id']: g for g in lab['boxes']}
        removed_trk = {m['trk'] for m in rows if m['removed'] and m['trk'] is not None}
        link = {m['trk']: gt_by[m['box_id']] for m in rows if not m['removed'] and m['trk'] is not None}
        for t in rec['tracks']:
            tid = int(t['tid'])
            if tid in removed_trk:
                continue
            k = _rep(t)
            if k is None:
                continue
            r = _row(split, vid, rec, lab, t, k, link.get(tid), len(lab['boxes']))
            alltrk.append(r)
        for m in rows:                                       # 매핑 쌍 표 (예전과 같은 순서: 매핑 엑셀 순)
            if m['removed'] or m['trk'] is None:
                continue
            t = next((t for t in rec['tracks'] if int(t['tid']) == m['trk']), None)
            k = _rep(t) if t is not None else None
            if k is not None:
                pairs.append(_row(split, vid, rec, lab, t, k, gt_by[m['box_id']], len(lab['boxes'])))
    return pd.DataFrame(pairs), pd.DataFrame(alltrk)


def scale_at(df, y):
    return rail_scale_at_y_vec(df.near_y.values, df.far_y.values, df.near_w.values, df.far_w.values, np.asarray(y, float))


def add_features(df, recs):
    """기하 특징 (추론 때 알 수 있는 값만): 레일 방향 sin · cos, log focal, log bbox 비율, log 픽셀당 cm"""
    d = df.copy().reset_index(drop=True)
    y_ref = d.y1 + BEST_ALPHA * (d.y2 - d.y1)
    d['w_base'] = d.bw * scale_at(d, y_ref)
    d['B_gt'] = d.w_gt / d.w_base
    sin_t, cos_t = [], []
    for _, r in d.iterrows():
        rec = recs[r.split][r.vid]; xv, yv = vanish_xy(rec['quad'])
        dx, dy = xv - (r.x1 + r.x2) / 2, r.y2 - yv
        if not (np.isfinite(dx) and np.isfinite(dy)) or dy <= 1:
            dx, dy = rec['vx'], -rec['vy']
            if dy < 0: dx, dy = -dx, -dy
        n = np.hypot(dx, dy) + 1e-9; sin_t.append(dx / n); cos_t.append(dy / n)
    d['sin_t'], d['cos_t'] = sin_t, cos_t
    d['logB'] = np.log(d.B_gt)
    d['log_s'] = np.log(scale_at(d, y_ref))
    d['log_f'] = np.log(d.focal_mm)
    d['log_aspect'] = np.log(d.bw / d.bh)
    d['log_dw'], d['log_hw'] = np.log(d.d_gt / d.w_gt), np.log(d.h_gt / d.w_gt)
    return d


def sort3(a):
    return -np.sort(-np.asarray(a, float), axis=-1)

In [ ]:
# [M-1] 공용 도구 — 학습 설정 · 지표 (① ② ③) · 출력 도우미
import torch, torch.nn as nn

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
BS, NO_STD = 32, {'sin_t', 'cos_t'}
CKPT2 = f'{CKPT_DIR}/step2'; os.makedirs(CKPT2, exist_ok=True)       # 최종 head 가중치 위치 (ablation 노트북과 같음)
GEO = ['log_f', 'sin_t', 'cos_t', 'log_aspect', 'log_s']             # STEP 1 보조 입력: log focal · 레일 방향 · log bbox 가로세로 비 · log 픽셀당 cm
FS, T2 = ['log_f', 'log_s'], ['log_dw', 'log_hw']                    # STEP 2 보조 입력 · 출력


def hr(title):
    print('\n' + '━' * 90); print(title)


def dim_err(P, G):
    # 지표 ② (상자 하나씩): 세 변 큰 순 정렬 후 같은 순위끼리 → (cm 평균, % 평균)
    Gs = sort3(G); d = np.abs(sort3(P) - Gs)
    return d.mean(1).mean(), (d / Gs).mean(1).mean() * 100


def full_errors(labels, mapping, pred, cols=('w', 'd', 'h')):
    # 반환: 매칭 쌍 · 미검출 정답 · 정답 없는 트랙 — 상자마다 (cm 오차, % 오차)
    cols = list(cols)
    pidx = pred.set_index(['vid', 'tid'])
    matched, miss, fp = [], [], []
    for vid in sorted(labels):
        rows = mapping.get(vid, [])
        removed_box = {m['box_id'] for m in rows if m['removed']}
        removed_trk = {m['trk'] for m in rows if m['removed'] and m['trk'] is not None}
        link = {m['box_id']: m['trk'] for m in rows if not m['removed'] and m['trk'] is not None}
        vid_trks = set(pred.loc[pred.vid == vid, 'tid'].astype(int)) - removed_trk
        for g in labels[vid]['boxes']:
            if g['box_id'] in removed_box:
                continue
            gs = sort3([g['w'], g['d'], g['h']]); t = link.get(g['box_id'])
            if t is not None and (vid, t) in pidx.index:
                diff = np.abs(sort3(pidx.loc[(vid, t), cols].values.astype(float)) - gs)
                matched.append((float(diff.mean()), float((diff / gs).mean() * 100)))
            else:
                miss.append((float(gs.mean()), 100.0))                          # 예측을 (0, 0, 0)으로 → 세 변 모두 100%
        for t in vid_trks - set(link.values()):
            p = sort3(pidx.loc[(vid, t), cols].values.astype(float))
            fp.append((float(p.mean()), 100.0))                                  # 정답을 (0, 0, 0)으로 → 100%로 정의
    return matched, miss, fp


def box_records(labels, mapping, pred, cols=('w', 'd', 'h')):
    """full_errors와 같은 정의, 상자마다 (영상, 종류, cm, %, 정답 가장 긴 변 · 짧은 변) — 구간별 · bootstrap용"""
    cols = list(cols); pidx = pred.set_index(['vid', 'tid']); out = []
    for vid in sorted(labels):
        rows = mapping.get(vid, [])
        removed_box = {m['box_id'] for m in rows if m['removed']}
        removed_trk = {m['trk'] for m in rows if m['removed'] and m['trk'] is not None}
        link = {m['box_id']: m['trk'] for m in rows if not m['removed'] and m['trk'] is not None}
        n_box = len(labels[vid]['boxes'])
        for g in labels[vid]['boxes']:
            if g['box_id'] in removed_box: continue
            gs = sort3([g['w'], g['d'], g['h']]); t = link.get(g['box_id'])
            if t is not None and (vid, t) in pidx.index:
                diff = np.abs(sort3(pidx.loc[(vid, t), cols].values.astype(float)) - gs)
                out.append((vid, n_box, '매칭', float(diff.mean()), float((diff / gs).mean() * 100), gs[0], gs[2]))
            else:
                out.append((vid, n_box, '미검출', float(gs.mean()), 100.0, gs[0], gs[2]))
        vid_trks = set(pred.loc[pred.vid == vid, 'tid'].astype(int)) - removed_trk
        for t in vid_trks - set(link.values()):
            p = sort3(pidx.loc[(vid, t), cols].values.astype(float))
            out.append((vid, n_box, '정답 없음', float(p.mean()), 100.0, np.nan, np.nan))
    return pd.DataFrame(out, columns=['vid', 'n_box', 'kind', 'cm', 'pct', 'long', 'short'])


def boot_ci(df, col, n=2000, seed=0):
    """영상 단위 bootstrap 95% CI (영상을 복원추출, 상자 평균)"""
    if not len(df): return (np.nan, np.nan)
    rng = np.random.default_rng(seed); vids = df.vid.unique(); g = {v: df.loc[df.vid == v, col].values for v in vids}
    res = [np.concatenate([g[v] for v in rng.choice(vids, len(vids))]).mean() for _ in range(n)]
    return tuple(np.percentile(res, [2.5, 97.5]))


# **2. 데이터** — 수작업 매핑 쌍 (학습 · 지표 ① ②) + 모든 트랙 (지표 ③) · 영상 단위 5-fold

In [ ]:
# [D-1] 라벨 · 캐시 · 매핑 → 매핑 쌍 표 + 모든 트랙 표, train 영상 단위 5-fold (예전 분할 파일 그대로)
LABELS = {sp: load_labels(f'{DATA_DIR}/{sp}_label.json') for sp in SPLITS}
RECS   = {sp: load_cache(f'{CACHE_DIR}/{sp}') for sp in SPLITS}
MAPS   = {sp: load_mapping(MAPPING_XLSX[sp]) for sp in SPLITS}
PAIRS, ALL = {}, {}
for sp in SPLITS:
    p, a = build_tables(RECS[sp], LABELS[sp], sp, MAPS[sp])
    PAIRS[sp], ALL[sp] = add_features(p, RECS), add_features(a, RECS)
    print(f'{sp}: 매핑 쌍 {len(PAIRS[sp])} | 모든 트랙 {len(ALL[sp])} (그중 정답 없는 트랙 {int((~ALL[sp].mapped).sum())})')
TRN = PAIRS['train']

N_FOLD, SEED = 5, 0
FOLD_CSV = f'{OUT_DIR}/step1_folds.csv'
_key = lambda d: (d.vid + '|' + d.tid.astype(str) + '|' + d.box_id.astype(str)).tolist()
if os.path.exists(FOLD_CSV):
    F = pd.read_csv(FOLD_CSV)
    assert _key(F) == _key(TRN), 'train 쌍이 예전 분할 파일과 다름 → 저장된 fold 가중치를 쓸 수 없음 (매핑 · 캐시 확인)'
    TRN['fold'] = F['fold'].values
else:                                                    # 파일이 없을 때만 새로 만듦 (이 경우 가중치도 새로 학습됨)
    from sklearn.model_selection import StratifiedGroupKFold
    TRN['fold'] = -1
    fbin = pd.cut(TRN.focal_mm, [0, 8, 15, 25, 100], labels=False)
    for k, (_, te) in enumerate(StratifiedGroupKFold(N_FOLD, shuffle=True, random_state=SEED).split(TRN, fbin, groups=TRN.vid)):
        TRN.loc[te, 'fold'] = k
    TRN[['vid', 'tid', 'box_id', 'fold']].to_csv(FOLD_CSV, index=False)
VID_FOLD = TRN.groupby('vid').fold.first().to_dict()
ALL['train']['fold'] = ALL['train'].vid.map(VID_FOLD).fillna(-1).astype(int)   # OOF: 그 영상이 빠진 fold의 모델로 예측
for sp in SPLITS:
    if sp != 'train':
        ALL[sp]['fold'] = -1                                                     # val · test: fold 모델 5개 평균
print('train fold별 영상 수:', TRN.groupby('fold').vid.nunique().to_dict())

📄 train_label.json: 영상 100개, 상자 918개
📄 val_label.json: 영상 49개, 상자 467개
📄 test_label.json: 영상 50개, 상자 476개
🗂 train_label_mapping.xlsx: 영상 100개 → 트랙과 연결된 GT 899개
🗂 val_label_mapping.xlsx: 영상 49개 → 트랙과 연결된 GT 461개
🗂 test_label_mapping.xlsx: 영상 50개 → 트랙과 연결된 GT 469개
train: 매핑 쌍 899 | 모든 트랙 910 (그중 정답 없는 트랙 11)
val: 매핑 쌍 461 | 모든 트랙 467 (그중 정답 없는 트랙 6)
test: 매핑 쌍 469 | 모든 트랙 477 (그중 정답 없는 트랙 8)
train fold별 영상 수: {0: 20, 1: 20, 2: 20, 3: 19, 4: 21}


# **3. 프레임 선택 · crop** — 트랙마다 가려짐 적은 순 최대 8장

In [ ]:
# [P-0] crop 도구 — letterbox · 캐시 저장 (여유 10%, letterbox 224)
IMG, PAD = 224, 0.10


def letterbox(img, size=IMG):
    h, w = img.shape[:2]; s = size / max(h, w)
    r = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    out = np.zeros((size, size, 3), np.uint8)
    oy, ox = (size - r.shape[0]) // 2, (size - r.shape[1]) // 2
    out[oy:oy + r.shape[0], ox:ox + r.shape[1]] = r
    return out


def safe_npz(path):
    # 저장이 중간에 끊겨 깨진 캐시는 지우고 None (다시 만듦)
    if not os.path.exists(path):
        return None
    try:
        z = np.load(path, allow_pickle=True); z.files; return z
    except Exception as e:
        print(f'⚠️ 깨진 캐시 삭제 후 다시 만듦: {os.path.basename(path)} ({type(e).__name__})'); os.remove(path); return None


def safe_save(path, **arrays):
    # 로컬에 먼저 저장한 뒤 Drive로 옮김 → 저장 중 끊겨도 Drive에 깨진 파일이 남지 않음
    tmp = f'/content/_tmp_{os.path.basename(path)}'
    np.savez_compressed(tmp, **arrays); shutil.move(tmp, path)


In [ ]:
# [P-1] 프레임 선택 (가려짐 기준) → 프레임 표 + crop 캐시 (매칭 트랙 · 정답 없는 트랙 따로)
K_NEW, NEAR_NEW, MIN_GAP = 8, 0.5, 2                 # 가려짐 적은 순 최대 8장 · 면적이 최대의 50% 이상 · 프레임 간격 2 이상
MODE = 'multi_occ_k8'


def occlusion(bb_self, others):
    if not len(others):
        return 0.0
    o = np.asarray(others, float)
    iw = np.clip(np.minimum(bb_self[2], o[:, 2]) - np.maximum(bb_self[0], o[:, 0]), 0, None)
    ih = np.clip(np.minimum(bb_self[3], o[:, 3]) - np.maximum(bb_self[1], o[:, 1]), 0, None)
    area = max((bb_self[2] - bb_self[0]) * (bb_self[3] - bb_self[1]), 1e-6)
    return float(min(1.0, (iw * ih).sum() / area))


def select_frames(bb, occ, k=K_NEW, near=NEAR_NEW):
    area = (bb[:, 2] - bb[:, 0]) * (bb[:, 3] - bb[:, 1])
    valid = (bb[:, 2] > bb[:, 0] + 3) & (bb[:, 3] > bb[:, 1] + 3)
    order = [int(i) for i in np.argsort(-area) if valid[i]]
    if not order:
        return []
    cand = sorted([i for i in order if area[i] >= near * area[order[0]]], key=lambda i: (round(occ[i], 3), -area[i]))
    sel = []
    for i in cand:
        if all(abs(i - j) >= MIN_GAP for j in sel):
            sel.append(i)
        if len(sel) == k:
            break
    return sel


def frame_table(split, mapped=True):
    # mapped=True: 수작업 매핑된 트랙 (학습 · 지표 ① ②) / False: 정답 없는 트랙 (지표 ③, REMOVED 제외)
    rows = []
    for vid, rec in RECS[split].items():
        if vid not in LABELS[split] or rec['quad'] is None:
            continue
        lab, maps = LABELS[split][vid], MAPS[split].get(vid, [])
        gt_by = {g['box_id']: g for g in lab['boxes']}
        link = {m['trk']: gt_by[m['box_id']] for m in maps if not m['removed'] and m['trk'] is not None and m['box_id'] in gt_by}
        removed_trk = {m['trk'] for m in maps if m['removed'] and m['trk'] is not None}
        at = {}                                                        # 샘플 프레임 → [(트랙, bbox)]
        for t in rec['tracks']:
            for fi, b in zip(t['fi'], t['bbox']):
                at.setdefault(int(fi), []).append((int(t['tid']), b))
        for t in rec['tracks']:
            tid = int(t['tid']); g = link.get(tid)
            if mapped and g is None:
                continue
            if not mapped and (g is not None or tid in removed_trk or _rep(t) is None):
                continue
            occ = np.array([occlusion(b, [o for t2, o in at[int(fi)] if t2 != tid]) for fi, b in zip(t['fi'], t['bbox'])])
            for j in select_frames(t['bbox'], occ):
                r = _row(split, vid, rec, lab, t, j, g, len(lab['boxes']))
                r['occ'] = occ[j]
                rows.append(r)
    return add_features(pd.DataFrame(rows), RECS) if rows else None


def build_crops_k(df, tag):
    # build_crops와 같은 crop, 키에 프레임 번호까지 포함 (한 트랙에 여러 프레임)
    path = f'{CACHE_DIR}/step13_crops_{df.split.iloc[0]}_{tag}_{IMG}_p{int(PAD * 100)}.npz'
    keys = (df.vid + '|' + df.tid.astype(str) + '|' + df.rep_fi.astype(str)).values
    z = safe_npz(path)
    if z is not None and len(z['keys']) == len(keys) and (z['keys'] == keys).all():
        return z['imgs']
    imgs = np.zeros((len(df), IMG, IMG, 3), np.uint8)
    for (sp, vid), g in df.groupby(['split', 'vid'], sort=False):
        cap = cv2.VideoCapture(video_path(sp, vid))
        for i, r in sorted(g.iterrows(), key=lambda z: z[1].rep_fi):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(r.rep_fi) * int(r.step)); ok_, fr = cap.read()
            if not ok_:
                continue
            H, W = fr.shape[:2]; px, py = r.bw * PAD, r.bh * PAD
            c = fr[int(max(0, r.y1 - py)):int(min(H, r.y2 + py)), int(max(0, r.x1 - px)):int(min(W, r.x2 + px))]
            imgs[i] = letterbox(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
        cap.release()
    safe_save(path, imgs=imgs, keys=keys)
    print(f'💾 {os.path.basename(path)}  ({len(df)}개)')
    return imgs


FR, FC = {}, {}
for sp in SPLITS:
    for part, mapped, tag in (('mapped', True, f'x{MODE}'), ('unmapped', False, f'x{MODE}_unmapped')):
        d = frame_table(sp, mapped)
        FR[(sp, part)], FC[(sp, part)] = d, (build_crops_k(d, tag) if d is not None else None)
        if d is None:
            print(f'{sp:5s} {part:8s}: 트랙 0'); continue
        n = d.groupby(['vid', 'tid']).ngroups
        print(f'{sp:5s} {part:8s}: 샘플 {len(d):5d} · 트랙 {n:4d} · 트랙당 평균 {len(d) / n:.1f}장 · 평균 가려짐 {d.occ.mean():.3f}')
    n_map = FR[(sp, 'mapped')].groupby(['vid', 'tid']).ngroups
    if n_map != len(PAIRS[sp]):
        print(f'  ⚠️ {sp}: 프레임 표의 매칭 트랙 {n_map} ≠ 매칭 쌍 {len(PAIRS[sp])} → 빠진 트랙은 [F-2]에서 train 중앙값으로 채움')

train mapped  : 샘플  7127 · 트랙  899 · 트랙당 평균 7.9장 · 평균 가려짐 0.412
train unmapped: 샘플    63 · 트랙   11 · 트랙당 평균 5.7장 · 평균 가려짐 0.345
val   mapped  : 샘플  3589 · 트랙  461 · 트랙당 평균 7.8장 · 평균 가려짐 0.364
val   unmapped: 샘플    39 · 트랙    6 · 트랙당 평균 6.5장 · 평균 가려짐 0.488
test  mapped  : 샘플  3658 · 트랙  469 · 트랙당 평균 7.8장 · 평균 가려짐 0.399
test  unmapped: 샘플    45 · 트랙    8 · 트랙당 평균 5.6장 · 평균 가려짐 0.382


# **4. 토큰 캐시** — DINOv3-L · DA3 인코더, 프레임 crop마다 원본 · 좌우반전 (TTA용)

In [ ]:
# [T-0] DINOv3-L 7 × 7 토큰 캐시 (프레임 crop마다 원본 · 좌우반전) — 캐시가 있으면 읽기만 (test만 처음 추출)
#  - timm 1.0.20 이상 필요 (DINOv3)
import torch.nn.functional as F
DINO_NAME, TOK_RES, GRID = 'vit_large_patch16_dinov3.lvd1689m', 448, 7
if not timm.list_models('vit_large_patch16_dinov3*'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'timm>=1.0.20'], check=True)
    raise RuntimeError('timm 업그레이드 완료 → 런타임 > 세션 다시 시작 후 처음부터 다시 실행')


@torch.no_grad()
def extract_tokens(model, mean, std, n_prefix, patch, crops, flip, bs=32, times=None):
    X = torch.from_numpy(crops).permute(0, 3, 1, 2); g = TOK_RES // patch; out = []
    for i in range(0, len(X), bs):
        if DEV == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        x = (X[i:i + bs].to(DEV).float() / 255 - mean) / std
        if flip: x = x.flip(3)
        x = F.interpolate(x, size=(TOK_RES, TOK_RES), mode='bilinear', align_corners=False)
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEV == 'cuda'):
            h = model.forward_features(x)
        pt = h.float()[:, n_prefix:]; assert pt.shape[1] == g * g, (pt.shape, g)
        gr = pt.transpose(1, 2).reshape(len(x), -1, g, g)
        out.append(F.adaptive_avg_pool2d(gr, GRID).flatten(2).transpose(1, 2).half().cpu())
        if DEV == 'cuda': torch.cuda.synchronize()
        if times is not None: times.append((time.perf_counter() - t0) / len(x))
    return torch.cat(out).numpy()


def tok_path(sp, part):
    return f'{CACHE_DIR}/step1d_dinov3L_{sp}_{part}_{TOK_RES}.npz'


TOK1, todo, DINO_TIMES = {}, [], []
for sp in SPLITS:
    for part in ('mapped', 'unmapped'):
        d = FR[(sp, part)]
        if d is None: continue
        d = d.reset_index(drop=True); keys = (d.vid + '|' + d.tid.astype(str) + '|' + d.rep_fi.astype(str)).values
        z = safe_npz(tok_path(sp, part))
        if z is not None and len(z['keys']) == len(keys) and (z['keys'] == keys).all():
            TOK1[(sp, part)] = {'p7': z['p7'], 'p7_f': z['p7_f']}
        else:
            todo.append((sp, part, keys))
if todo:
    DINO = timm.create_model(DINO_NAME, pretrained=True, num_classes=0, img_size=TOK_RES).to(DEV).eval()
    _dc = timm.data.resolve_model_data_config(DINO)
    _mean, _std = (torch.tensor(_dc[k], device=DEV).view(1, 3, 1, 1) for k in ('mean', 'std'))
    _npre, _patch = int(DINO.num_prefix_tokens), int(DINO.patch_embed.patch_size[0])
    print(f'{DINO_NAME}: {sum(p.numel() for p in DINO.parameters()) / 1e6:.1f}M · patch {_patch} · 앞 토큰 {_npre}')
    for sp, part, keys in todo:
        p, pf = (extract_tokens(DINO, _mean, _std, _npre, _patch, FC[(sp, part)], fl, times=DINO_TIMES) for fl in (False, True))
        safe_save(tok_path(sp, part), keys=keys, p7=p, p7_f=pf); TOK1[(sp, part)] = {'p7': p, 'p7_f': pf}
        print(f'  💾 {os.path.basename(tok_path(sp, part))} {p.shape}')
    del DINO; torch.cuda.empty_cache()
    print(f'DINOv3-L crop당 {1000 * np.median(DINO_TIMES):.1f} ms (배치 32, 448 입력, 원본 · 반전 포함 측정)')
for k, v in TOK1.items():
    print(f'{k[0]:5s} {k[1]:8s}: 토큰 {v["p7"].shape}')

train mapped  : 토큰 (7127, 49, 1024)
train unmapped: 토큰 (63, 49, 1024)
val   mapped  : 토큰 (3589, 49, 1024)
val   unmapped: 토큰 (39, 49, 1024)
test  mapped  : 토큰 (3658, 49, 1024)
test  unmapped: 토큰 (45, 49, 1024)


In [ ]:
# [T-0b] DA3 인코더 7 × 7 토큰 캐시 (프레임 crop마다 원본 · 좌우반전) — 캐시가 있으면 읽기만
#  - Depth Anything 3 (DA3METRIC-LARGE) 안의 ViT 인코더(DinoVisionTransformer, 304.4 M, 고정)의 마지막 층 패치 토큰
#  - DINOv3와 같은 방식: crop 224 → 448 (patch 14 → 32 × 32) → 7 × 7 평균 pooling → (49, 1024)
#  - train · val 캐시는 ablation [2C-13]과 같은 파일 (step2c13_da3L_*) → test만 새로 추출 (A100에서 수 분)
#  - DA3 설치 시 moviepy · pkg_resources · gsplat 경고가 나와도 무시해도 된다
import importlib, types, gc


def da3_path(sp, part):
    return f'{CACHE_DIR}/step2c13_da3L_{sp}_{part}_{TOK_RES}.npz'


def ensure_da3():
    if 'DA3' in globals():
        return DA3
    try:
        import depth_anything_3  # noqa
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--ignore-requires-python',
                        'git+https://github.com/ByteDance-Seed/Depth-Anything-3.git'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'addict', 'einops', 'omegaconf', 'safetensors',
                        'imageio', 'trimesh', 'plyfile'], check=True)
    for n in ('moviepy.editor', 'pycolmap', 'evo.core.trajectory'):          # 단안 깊이에 안 쓰는 모듈은 빈 모듈로
        try:
            importlib.import_module(n)
        except Exception:
            parts = n.split('.')
            for i in range(1, len(parts) + 1):
                k = '.'.join(parts[:i])
                if k not in sys.modules:
                    m = types.ModuleType(k); m.__getattr__ = lambda a: None; sys.modules[k] = m
    from depth_anything_3.api import DepthAnything3
    globals()['DA3'] = DepthAnything3.from_pretrained('depth-anything/DA3METRIC-LARGE').to(DEV).eval()
    return DA3


def da3_encoder():
    da3 = ensure_da3()
    return min([(n, m) for n, m in da3.named_modules() if hasattr(m, 'get_intermediate_layers')], key=lambda c: len(c[0]))[1]


IM_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
IM_STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)
DA3_G = TOK_RES // 14


@torch.no_grad()
def da3_tokens_gpu(enc, x01):
    """x01: (B, 3, 448, 448), 0 ~ 1 → (B, 49, 1024) GPU. DA3 인코더는 (B, 장 수, 3, H, W)를 받음 → 장 수 = 1"""
    with torch.autocast('cuda', dtype=torch.float16, enabled=DEV == 'cuda'):
        o = enc.get_intermediate_layers(((x01 - IM_MEAN) / IM_STD).unsqueeze(1), n=1)
    h = o[0][0][0][:, 0].float()                                             # (B, 1, 1024, 1024) → (B, 1024 패치, 1024)
    gr = h[:, -DA3_G * DA3_G:].transpose(1, 2).reshape(len(h), -1, DA3_G, DA3_G)
    return F.adaptive_avg_pool2d(gr, GRID).flatten(2).transpose(1, 2)


@torch.no_grad()
def da3_tokens(enc, crops, flip, bs=64):
    X = torch.from_numpy(crops).permute(0, 3, 1, 2); out = []
    for i in range(0, len(X), bs):
        x = X[i:i + bs].to(DEV).float() / 255
        if flip: x = x.flip(3)
        x = F.interpolate(x, size=(DA3_G * 14, DA3_G * 14), mode='bilinear', align_corners=False)
        out.append(da3_tokens_gpu(enc, x).half().cpu())
    return torch.cat(out).numpy()


TOK_DA3, _enc = {}, None
for sp in SPLITS:
    for part in ('mapped', 'unmapped'):
        d = FR[(sp, part)]
        if d is None: continue
        d = d.reset_index(drop=True); keys = (d.vid + '|' + d.tid.astype(str) + '|' + d.rep_fi.astype(str)).values
        z = safe_npz(da3_path(sp, part))
        if z is not None and len(z['keys']) == len(keys) and (z['keys'] == keys).all():
            TOK_DA3[(sp, part)] = {'p7': z['p7'], 'p7_f': z['p7_f']}; continue
        if _enc is None: _enc = da3_encoder()
        t0 = time.time(); p, pf = da3_tokens(_enc, FC[(sp, part)], False), da3_tokens(_enc, FC[(sp, part)], True)
        safe_save(da3_path(sp, part), keys=keys, p7=p, p7_f=pf); TOK_DA3[(sp, part)] = {'p7': p, 'p7_f': pf}
        print(f'  💾 {os.path.basename(da3_path(sp, part))} {p.shape} · {time.time() - t0:.0f}s')
del _enc; gc.collect(); torch.cuda.empty_cache()
for k, v in TOK_DA3.items():
    print(f'{k[0]:5s} {k[1]:8s}: DA3 토큰 {v["p7"].shape}')


def tok_cat(sp, part):
    """최종 입력: 칸마다 DA3 토큰 · DINOv3 토큰을 이어 붙임 (49, 2048)"""
    a, b = TOK_DA3[(sp, part)], TOK1[(sp, part)]
    return np.concatenate([a['p7'], b['p7']], -1), np.concatenate([a['p7_f'], b['p7_f']], -1)

train mapped  : DA3 토큰 (7127, 49, 1024)
train unmapped: DA3 토큰 (63, 49, 1024)
val   mapped  : DA3 토큰 (3589, 49, 1024)
val   unmapped: DA3 토큰 (39, 49, 1024)
test  mapped  : DA3 토큰 (3658, 49, 1024)
test  unmapped: DA3 토큰 (45, 49, 1024)


# **5. 최종 모델** — 고정 토큰 (DA3 ⊕ DINOv3) 위의 STEP 1 head · STEP 2 head

In [ ]:
# [F-1] 최종 head · 학습 · 예측 도구
class TokHead(nn.Module):
    def __init__(self, init_bias, n_side, n_tok, tok_dim, proj=64):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(tok_dim), nn.Linear(tok_dim, proj), nn.GELU())
        self.head = nn.Sequential(nn.Linear(n_tok * proj + n_side, 256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, len(init_bias)))
        nn.init.zeros_(self.head[-1].weight)
        with torch.no_grad():
            self.head[-1].bias.copy_(torch.tensor(init_bias, dtype=torch.float32))

    def forward(self, t, s):
        return self.head(torch.cat([self.proj(t).flatten(1), s], 1))


STEP1 = dict(name='2C13_da3L_v3L_step1tok', side=GEO, tgt=['logB'], epochs=40)      # ablation [2C-13]과 같은 가중치 이름
STEP2 = dict(name='2C13_da3L_v3L_step2multi', side=FS, tgt=T2, epochs=25)


def _std(df, side, rows):
    A = df[side].values.astype(np.float32); mu, sd = A[rows].mean(0), A[rows].std(0) + 1e-6
    for j, c in enumerate(side):
        if c in NO_STD: mu[j], sd[j] = 0.0, 1.0
    return mu, sd


def train_head(cfg, df, T, Tf, fold):
    name, side, tgt = cfg['name'], list(cfg['side']), list(cfg['tgt'])
    if all(os.path.exists(f'{CKPT2}/{name}_fold{k}.pt') for k in range(N_FOLD)):
        print(f'  [{name}] fold 가중치 5개 있음 → 학습 건너뜀'); return
    sin_i = side.index('sin_t') if 'sin_t' in side else None
    Tg, Tfg = torch.as_tensor(T).to(DEV), torch.as_tensor(Tf).to(DEV)
    Y = torch.tensor(df[tgt].values, dtype=torch.float32); A = torch.from_numpy(df[side].values.astype(np.float32))
    loss_fn, t0 = nn.SmoothL1Loss(beta=0.1), time.time()
    for k in range(N_FOLD):
        path = f'{CKPT2}/{name}_fold{k}.pt'
        if os.path.exists(path): continue
        tr = np.where(fold != k)[0]
        mu, sd = _std(df, side, tr); S = (A - torch.from_numpy(mu)) / torch.from_numpy(sd)
        torch.manual_seed(SEED + k); rng = np.random.default_rng(SEED + k)
        model = TokHead(np.median(Y[tr].numpy(), 0), len(side), Tg.shape[1], Tg.shape[2]).to(DEV)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
        n_step = cfg['epochs'] * math.ceil(len(tr) / BS); warm = n_step // 20
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda i: min(1, (i + 1) / max(warm, 1)) *
                                                  0.5 * (1 + math.cos(math.pi * min(i / n_step, 1))))
        for ep in range(cfg['epochs']):
            model.train(); perm = rng.permutation(tr)
            for i in range(0, len(perm), BS):
                idx = perm[i:i + BS]
                if len(idx) < 2: continue
                it = torch.from_numpy(idx).to(DEV); fl = torch.rand(len(idx), device=DEV) < 0.5
                t = torch.where(fl[:, None, None], Tfg[it], Tg[it]).float(); s = S[idx].to(DEV)
                if sin_i is not None:
                    s = s.clone(); s[fl, sin_i] *= -1
                loss = loss_fn(model(t, s), Y[idx].to(DEV))
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        torch.save({'state': model.state_dict(), 'mu': mu, 'sd': sd, 'cfg': cfg}, path)
        del model, opt
    print(f'  [{name}] 학습 {time.time() - t0:.0f}s'); del Tg, Tfg; torch.cuda.empty_cache()


@torch.no_grad()
def infer_head(cfg, df, T, Tf, fold, bs=512):
    side, n_out = list(cfg['side']), len(cfg['tgt'])
    sin_i = side.index('sin_t') if 'sin_t' in side else None
    T, Tf = torch.as_tensor(T), torch.as_tensor(Tf)
    A = torch.from_numpy(df[side].values.astype(np.float32)); out = np.zeros((len(df), n_out))
    for k in range(N_FOLD):
        rows = np.where((fold == k) | (fold < 0))[0]
        if not len(rows): continue
        ck = torch.load(f"{CKPT2}/{cfg['name']}_fold{k}.pt", map_location='cpu', weights_only=False)
        S = (A[rows] - torch.from_numpy(ck['mu'])) / torch.from_numpy(ck['sd'])
        model = TokHead(np.zeros(n_out), len(side), T.shape[1], T.shape[2]).to(DEV)
        model.load_state_dict(ck['state']); model.eval(); p = []
        for i in range(0, len(rows), bs):
            r = torch.from_numpy(rows[i:i + bs]); s = S[i:i + bs].to(DEV); sf = s.clone()
            if sin_i is not None: sf[:, sin_i] *= -1
            p.append(((model(T[r].to(DEV).float(), s) + model(Tf[r].to(DEV).float(), sf)) / 2).cpu())   # 좌우반전 TTA
        p = torch.cat(p).numpy(); oof = fold[rows] == k
        out[rows[oof]] = p[oof]; out[rows[~oof]] += p[~oof] / N_FOLD
    return out


N_HEAD = sum(p.numel() for p in TokHead(np.zeros(1), len(GEO), 49, 2048).parameters()) + \
         sum(p.numel() for p in TokHead(np.zeros(2), len(FS), 49, 2048).parameters())
print(f'학습 파라미터: STEP 1 + STEP 2 head {N_HEAD / 1e6:.2f} M (DA3 인코더 304.4 M · DINOv3-L 303.1 M는 고정)')

학습 파라미터: STEP 1 + STEP 2 head 1.88 M (DA3 인코더 304.4 M · DINOv3-L 303.1 M는 고정)


In [ ]:
# [F-2] 학습 (가중치 있으면 건너뜀) → 프레임별 예측 → 트랙마다 평균 → 모든 트랙 예측표 PRED[sp]
#  - 입력 토큰 = [T-0b] tok_cat (DA3 + DINOv3, 49 × 2048). 가중치는 ablation [2C-13]과 같은 이름이라 있으면 학습을 건너뜀
#  - 저장: OUT2_DIR/{split}_proposed_preds.csv (vid, tid, mapped, w, d, h)
FRT = FR[('train', 'mapped')].reset_index(drop=True)
FOLD_FR = FRT.vid.map(VID_FOLD).astype(int).values
hr('▶ 최종 head 학습 (가중치가 있으면 건너뜀)')
if not all(os.path.exists(f"{CKPT2}/{c['name']}_fold{k}.pt") for c in (STEP1, STEP2) for k in range(N_FOLD)):
    _T, _Tf = tok_cat('train', 'mapped')
    for cfg in (STEP1, STEP2):
        train_head(cfg, FRT, _T, _Tf, FOLD_FR)
    del _T, _Tf; gc.collect()
else:
    print('  STEP 1 · STEP 2 fold 가중치 10개 있음 → 학습 건너뜀')

FALLBACK = {'logB': float(np.median(FRT.logB)), 'log_dw': float(np.median(FRT.log_dw)), 'log_hw': float(np.median(FRT.log_hw))}
PRED, FRAME_PRED = {}, {}
for sp in SPLITS:
    parts = []
    for part in ('mapped', 'unmapped'):
        f = FR[(sp, part)]
        if f is None: continue
        f = f.reset_index(drop=True); T, Tf = tok_cat(sp, part)
        fold = f.vid.map(VID_FOLD).fillna(-1).astype(int).values if sp == 'train' else np.full(len(f), -1)
        p1 = infer_head(STEP1, f, T, Tf, fold)[:, 0]
        p2 = infer_head(STEP2, f, T, Tf, fold)
        parts.append(f[['vid', 'tid', 'rep_fi', 'occ']].assign(lw=np.log(f.w_base.values) + p1, ldw=p2[:, 0], lhw=p2[:, 1]))
        del T, Tf
    FRAME_PRED[sp] = pd.concat(parts, ignore_index=True)
    trk = FRAME_PRED[sp].groupby(['vid', 'tid']).agg(lw=('lw', 'mean'), ldw=('ldw', 'mean'), lhw=('lhw', 'mean'),
                                                      n_frames=('lw', 'size')).reset_index()
    D = ALL[sp][['split', 'vid', 'tid', 'box_id', 'mapped', 'n_gt', 'w_base', 'w_gt', 'd_gt', 'h_gt', 'focal_mm', 'sin_t', 'cos_t']]
    P = D.merge(trk, on=['vid', 'tid'], how='left')
    miss = P.lw.isna()
    if miss.any():                                                    # 프레임이 하나도 없는 트랙 (지금까지 0개)
        print(f'  ⚠️ {sp}: 프레임 없는 트랙 {int(miss.sum())}개 → train 중앙값 B · 형태비로 채움')
        P.loc[miss, 'lw'] = np.log(P.loc[miss, 'w_base']) + FALLBACK['logB']
        P.loc[miss, 'ldw'], P.loc[miss, 'lhw'] = FALLBACK['log_dw'], FALLBACK['log_hw']
    P['w'] = np.exp(P.lw); P['d'] = P.w * np.exp(P.ldw); P['h'] = P.w * np.exp(P.lhw)
    PRED[sp] = P
    P[['vid', 'tid', 'mapped', 'w', 'd', 'h']].to_csv(f'{OUT2_DIR}/{sp}_proposed_preds.csv', index=False)
    print(f'{sp}: 트랙 {len(P)}개 (매칭 {int(P.mapped.sum())}) · 트랙당 프레임 평균 {P.n_frames.mean():.1f} → {sp}_proposed_preds.csv')


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ 최종 head 학습 (가중치가 있으면 건너뜀)
  STEP 1 · STEP 2 fold 가중치 10개 있음 → 학습 건너뜀
train: 트랙 910개 (매칭 899) · 트랙당 프레임 평균 7.9 → train_proposed_preds.csv
val: 트랙 467개 (매칭 461) · 트랙당 프레임 평균 7.8 → val_proposed_preds.csv
test: 트랙 477개 (매칭 469) · 트랙당 프레임 평균 7.8 → test_proposed_preds.csv


# **6. train · val 채점** — 재현 확인 (val ② 1.952 cm) · 오차 몫 분해 · 변별 오차 · 구간별

In [ ]:
# [E-1] train · val 채점 (test는 [T-1]에서 한 번만)
SB_SHORT, SL_SHORT = [0, 10, 15, np.inf], ['≤ 10 cm', '10 ~ 15 cm', '> 15 cm']
SB_LONG, SL_LONG = [0, 30, 40, 50, np.inf], ['~ 30 cm', '30 ~ 40 cm', '40 ~ 50 cm', '50 cm ~']
SB_SCENE, SL_SCENE = [0, 5, 10, np.inf], ['2 ~ 5개', '6 ~ 10개', '11개 ~']


def summarize(sp):
    P = PRED[sp]; m = P[P.mapped]
    G = m[['w_gt', 'd_gt', 'h_gt']].values; wg, dg, hg = G.T; w = m.w.values
    R = box_records(LABELS[sp], MAPS[sp], P); mt = R[R.kind == '매칭']
    ew = np.abs(w - wg)
    row = {'① w cm': ew.mean(), '① w %': (ew / wg).mean() * 100, '① 혼잡 장면 w %': (ew / wg)[m.n_gt.values >= 11].mean() * 100,
           '② cm': mt.cm.mean(), '② %': mt.pct.mean(), '③ cm': R.cm.mean(), '③ %': R.pct.mean(),
           '매칭': len(mt), '미검출': int((R.kind == '미검출').sum()), '정답 없는 트랙': int((R.kind == '정답 없음').sum())}
    lb = dim_err(np.c_[w, w * dg / wg, w * hg / wg], G); s2 = dim_err(np.c_[wg, wg * np.exp(m.ldw), wg * np.exp(m.lhw)], G)
    dec = {'STEP 1만 cm': lb[0], 'STEP 1만 %': lb[1], 'STEP 2만 cm': s2[0], 'STEP 2만 %': s2[1],
           'STEP 1 몫 %': 100 * lb[1] / row['② %'],
           'corr(w, d/w)': np.corrcoef(np.log(w / wg), m.ldw.values - np.log(dg / wg))[0, 1],
           'corr(w, h/w)': np.corrcoef(np.log(w / wg), m.lhw.values - np.log(hg / wg))[0, 1]}
    Gs, Ps = sort3(G), sort3(m[['w', 'd', 'h']].values); d = np.abs(Ps - Gs)
    side = {f'{s} (정답 평균 {Gs[:, j].mean():.1f})': f'{d[:, j].mean():.2f} cm · {(d[:, j] / Gs[:, j]).mean() * 100:.1f}%'
            for j, s in enumerate(['긴 변', '중간 변', '짧은 변'])}
    return row, dec, side, R


hr('▶ train · val — 지표 ① ② ③ (cm · %)')
OUT_E = {sp: summarize(sp) for sp in SPLITS if sp != 'test'}
display(pd.DataFrame({sp: v[0] for sp, v in OUT_E.items()}).T.round(3))
hr('▶ 오차 몫 분해 (지표 ②)')
display(pd.DataFrame({sp: v[1] for sp, v in OUT_E.items()}).T.round(3))
hr('▶ 변별 오차 — 세 변을 큰 순으로 정렬해 같은 순위끼리')
display(pd.DataFrame({sp: v[2] for sp, v in OUT_E.items()}).T)


def bin_table(R, by, bins, labels, kinds=('매칭',)):
    r = R[R.kind.isin(kinds)].copy(); r['bin'] = pd.cut(r[by], bins, labels=labels)
    rows = []
    for b in labels:
        x = r[r.bin == b]; lo, hi = boot_ci(x, 'cm'); plo, phi = boot_ci(x, 'pct')
        rows.append({'구간': b, '상자 수': len(x), '영상 수': x.vid.nunique(), 'cm': x.cm.mean(), 'cm 95% CI': f'[{lo:.2f}, {hi:.2f}]',
                     '%': x.pct.mean(), '% 95% CI': f'[{plo:.1f}, {phi:.1f}]'})
    return pd.DataFrame(rows).set_index('구간')


for sp, (_, _, _, R) in OUT_E.items():
    hr(f'▶ {sp} — 짧은 변 구간 (지표 ②) · 가장 긴 변 구간 (지표 ②) · 장면 복잡도 (지표 ② · ③)')
    display(bin_table(R, 'short', SB_SHORT, SL_SHORT).round(2))
    display(bin_table(R, 'long', SB_LONG, SL_LONG).round(2))
    display(bin_table(R, 'n_box', SB_SCENE, SL_SCENE).round(2))
    display(bin_table(R, 'n_box', SB_SCENE, SL_SCENE, kinds=('매칭', '미검출', '정답 없음')).round(2).add_prefix('③ '))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ train · val — 지표 ① ② ③ (cm · %)


,① w cm,① w %,① 혼잡 장면 w %,② cm,② %,③ cm,③ %,매칭,미검출,정답 없는 트랙
train,0.838,3.427,3.878,1.795,9.201,2.043,10.984,899.0,7.0,11.0
val,1.253,5.178,7.296,1.952,10.311,2.439,12.586,461.0,6.0,6.0



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ 오차 몫 분해 (지표 ②)


,STEP 1만 cm,STEP 1만 %,STEP 2만 cm,STEP 2만 %,STEP 1 몫 %,"corr(w, d/w)","corr(w, h/w)"
train,0.741,3.427,1.66,8.863,37.246,-0.359,-0.391
val,1.000,5.178,1.86,10.561,50.216,-0.656,-0.582



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ 변별 오차 — 세 변을 큰 순으로 정렬해 같은 순위끼리


,긴 변 (정답 평균 31.9),중간 변 (정답 평균 21.8),짧은 변 (정답 평균 14.1),긴 변 (정답 평균 31.5),짧은 변 (정답 평균 14.2)
train,2.20 cm · 7.1%,1.56 cm · 8.0%,1.63 cm · 12.5%,NaN,NaN
val,NaN,1.72 cm · 9.5%,NaN,2.44 cm · 8.3%,1.70 cm · 13.2%


... (6 outputs left out)

,상자 수,영상 수,cm,cm 95% CI,%,% 95% CI
구간,,,,,,
≤ 10 cm,97,40,1.95,"[1.61, 2.32]",15.39,"[12.5, 18.6]"
10 ~ 15 cm,187,47,1.89,"[1.60, 2.18]",10.03,"[8.7, 11.4]"
> 15 cm,177,46,2.02,"[1.76, 2.31]",7.83,"[6.8, 9.0]"


,상자 수,영상 수,cm,cm 95% CI,%,% 95% CI
구간,,,,,,
~ 30 cm,225,48,1.85,"[1.55, 2.20]",12.35,"[10.2, 14.7]"
30 ~ 40 cm,138,45,1.80,"[1.61, 1.99]",8.41,"[7.4, 9.5]"
40 ~ 50 cm,71,36,2.34,"[1.89, 2.86]",8.38,"[7.1, 9.8]"
50 cm ~,27,21,2.56,"[2.02, 3.15]",8.10,"[6.9, 9.3]"


,상자 수,영상 수,cm,cm 95% CI,%,% 95% CI
구간,,,,,,
2 ~ 5개,39,9,1.61,"[1.31, 2.00]",8.25,"[6.4, 10.9]"
6 ~ 10개,181,22,1.69,"[1.50, 1.89]",8.68,"[7.8, 9.6]"
11개 ~,241,18,2.20,"[1.88, 2.58]",11.87,"[9.8, 14.1]"


,③ 상자 수,③ 영상 수,③ cm,③ cm 95% CI,③ %,③ % 95% CI
구간,,,,,,
2 ~ 5개,41,9,2.69,"[1.32, 4.57]",12.73,"[6.6, 21.4]"
6 ~ 10개,184,22,2.05,"[1.66, 2.53]",10.17,"[8.6, 12.0]"
11개 ~,248,18,2.68,"[2.04, 3.50]",14.35,"[10.6, 18.8]"


# **7. test 채점 (한 번만)**
- 모델 · 설정은 train · val로만 골랐고, test는 [T-1]에서 **한 번만** 채점한다. 채점 직전 val ② 1.952 cm 재현을 확인한다
- [T-2]는 같은 채점 함수로 OVMono3D-LIFT(원본 · 미세조정 A-1)와 비교한다 (baseline 예측 파일이 있을 때만)

In [ ]:
# [T-1] test 채점 + result.json — ⚠️ 한 번만
MARK = f'{OUT2_DIR}/TEST_SCORED.txt'
if 'test' in SPLITS:
    # 안전장치 1: 정리된 코드가 ablation의 최종 모델과 같은지 — val ②가 ablation [2C-13] 값(1.952 cm)과 같아야 test 채점
    _v = OUT_E['val'][0]['② cm']
    assert abs(_v - 1.952) < 0.01, f'val ② {_v:.3f} cm ≠ 1.952 cm → 가중치 · 캐시를 확인하기 전에는 test를 채점하지 않음'
    print(f'✅ 재현 확인: val ② {_v:.3f} cm (기대 1.952) → test 채점')
    # 안전장치 2: 이미 채점했으면 알림 (결과는 같지만, 채점 후 모델을 바꾸지 않았는지 확인용)
    if os.path.exists(MARK):
        print('⚠️ test는 이미 채점된 적이 있음:', open(MARK).read().strip())
    row, dec, side, R = summarize('test'); ROW_TEST = row
    lo, hi = boot_ci(R[R.kind == '매칭'], 'cm'); tlo, thi = boot_ci(R, 'cm')
    hr('▶ TEST — 지표 ① ② ③')
    display(pd.DataFrame({'test': {**row, '② cm 95% CI': f'[{lo:.2f}, {hi:.2f}]', '③ cm 95% CI': f'[{tlo:.2f}, {thi:.2f}]'}}).T)
    hr('▶ TEST — 오차 몫 분해 · 변별 오차')
    display(pd.DataFrame({'test': dec}).T.round(3)); display(pd.DataFrame({'test': side}).T)
    hr('▶ TEST — 가장 긴 변 구간 (지표 ②, cm · % · 영상 단위 bootstrap 95% CI)')
    display(bin_table(R, 'long', SB_LONG, SL_LONG).round(2))
    hr('▶ TEST — 장면 복잡도 (영상당 상자 수) · 지표 ② · ③')
    display(bin_table(R, 'n_box', SB_SCENE, SL_SCENE).round(2))
    display(bin_table(R, 'n_box', SB_SCENE, SL_SCENE, kinds=('매칭', '미검출', '정답 없음')).round(2).add_prefix('③ '))
    hr('▶ TEST — 짧은 변 구간 (지표 ②)')
    display(bin_table(R, 'short', SB_SHORT, SL_SHORT).round(2))

    P = PRED['test']; vids = sorted(LABELS['test'])
    result = {'videos': [{'video_id': v, 'objects': [{'size_cm': {'w': round(float(r.w), 3), 'd': round(float(r.d), 3), 'h': round(float(r.h), 3)}}
                                                     for r in P[P.vid == v].sort_values('tid').itertuples()]} for v in vids]}
    with open(f'{OUT2_DIR}/result.json', 'w') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    with open(MARK, 'a') as f:
        f.write(f'{time.strftime("%Y-%m-%d %H:%M")} test 채점 · ② {row["② cm"]:.3f} cm · ③ {row["③ cm"]:.3f} cm\n')
    print(f'💾 result.json: 영상 {len(vids)}개 · 상자 {sum(len(v["objects"]) for v in result["videos"])}개 → {OUT2_DIR}/result.json')
else:
    print('⏭ RUN_TEST = False → test 채점 건너뜀 (최종 평가 때 [S-1]에서 True로 바꾸고 처음부터 한 번만 실행)')

✅ 재현 확인: val ② 1.952 cm (기대 1.952) → test 채점
⚠️ test는 이미 채점된 적이 있음: 2026-10-28 15:36 test 채점 · ② 1.819 cm · ③ 2.222 cm
2026-10-31 12:34 test 채점 · ② 1.751 cm · ③ 2.155 cm
2026-10-31 13:06 test 채점 · ② 1.751 cm · ③ 2.155 cm

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ TEST — 지표 ① ② ③


,① w cm,① w %,① 혼잡 장면 w %,② cm,② %,③ cm,③ %,매칭,미검출,정답 없는 트랙,② cm 95% CI,③ cm 95% CI
test,1.024105,4.259794,5.744627,1.750553,8.909081,2.155411,11.732146,469,7,8,"[1.58, 1.93]","[1.81, 2.61]"



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ TEST — 오차 몫 분해 · 변별 오차


,STEP 1만 cm,STEP 1만 %,STEP 2만 cm,STEP 2만 %,STEP 1 몫 %,"corr(w, d/w)","corr(w, h/w)"
test,0.845,4.26,1.716,9.402,47.814,-0.712,-0.609


,긴 변 (정답 평균 32.3),중간 변 (정답 평균 21.6),짧은 변 (정답 평균 14.1)
test,2.19 cm · 7.0%,1.54 cm · 7.9%,1.51 cm · 11.8%



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ TEST — 가장 긴 변 구간 (지표 ②, cm · % · 영상 단위 bootstrap 95% CI)


... (3 outputs left out)

,③ 상자 수,③ 영상 수,③ cm,③ cm 95% CI,③ %,③ % 95% CI
구간,,,,,,
2 ~ 5개,38,9,1.78,"[1.24, 2.54]",11.00,"[7.3, 17.1]"
6 ~ 10개,172,21,1.84,"[1.54, 2.18]",10.26,"[8.3, 12.5]"
11개 ~,274,20,2.41,"[1.84, 3.09]",12.76,"[9.9, 16.0]"



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ TEST — 짧은 변 구간 (지표 ②)


,상자 수,영상 수,cm,cm 95% CI,%,% 95% CI
구간,,,,,,
≤ 10 cm,110,45,1.80,"[1.42, 2.27]",12.42,"[10.4, 14.6]"
10 ~ 15 cm,181,50,1.76,"[1.54, 1.99]",9.02,"[8.0, 10.1]"
> 15 cm,178,44,1.71,"[1.50, 1.92]",6.63,"[5.7, 7.5]"


💾 result.json: 영상 50개 · 상자 477개 → /content/drive/MyDrive/4학년 1학기 과목별 정리/소프트웨어융합캡스톤디자인/BoxSizing/outputs/step2/result.json


In [ ]:
# [T-2] test — 같은 채점 함수로 baseline과 비교 (같은 bbox · 대표 프레임 · focal, baseline 예측 파일이 있을 때만)
#  - 제안 (최종): PRED['test']가 있으면 그대로, 없으면 [F-2]가 저장한 OUT2_DIR/test_proposed_preds.csv를 읽음 (다시 학습 · 추론하지 않음)
#  - LIFT 원본 · LIFT + 레일 보정: OVMono3D 노트북의 raw pkl (outputs/baseline/ovmono3d_rep/raw_test), 레일 배율 s는 OVMono3D 노트북 [E-1]과 같은 정의
#  - LIFT 미세조정 A-1 · A-1R(미세조정 + 레일): ablation 노트북 [X-1] · [H-3]이 저장한 test 예측 (outputs/binpack/extra/)
#  - 영상 단위 paired bootstrap: (baseline − 최종), 양수 = 최종이 더 좋음
EXP_DIR = f'{PROJECT_DIR}/outputs/binpack'; EXTRA_DIR = f'{EXP_DIR}/extra'
OV_RAW = f'{PROJECT_DIR}/outputs/baseline/ovmono3d_rep'


def lift_raw(sp, rail=False):
    """(vid, tid) → LIFT 치수 cm. rail=True면 상자마다 레일 배율 s를 곱함 (GT 미사용)"""
    out, S = {}, {}
    for p in sorted(glob.glob(f'{OV_RAW}/raw_{sp}/*.pkl')):
        with open(p, 'rb') as f:
            r = pickle.load(f)
        q, f_px = r.get('quad'), r.get('f_px')
        for o in r.get('lift', []):
            k = (r['video_id'], int(o['tid']))
            if k in out:                                   # 중복 트랙은 첫 행 (OVMono3D 노트북 drop_duplicates와 같음)
                continue
            out[k] = np.asarray(o['dims_m'], float) * 100
            implied = o['z_m'] * 100 / f_px if f_px else 0
            S[k] = (float(rail_scale_at_y_vec(q['near_y'], q['far_y'], q['near_w'], q['far_w'], o['bbox'][3])) / implied
                    if (q is not None and implied > 1e-6) else np.nan)
    if rail and out:
        med = float(np.nanmedian(list(S.values())))
        out = {k: v * (S[k] if np.isfinite(S[k]) else med) for k, v in out.items()}
    return out


def extra_csv(name):
    _p = f'{EXTRA_DIR}/{name}'
    if not os.path.exists(_p):
        return None
    e = pd.read_csv(_p)
    return {(v, int(t)): (a, b, c) for v, t, a, b, c in e[['vid', 'tid', 'w', 'd', 'h']].itertuples(index=False)}


if 'test' in SPLITS:
    # ---- 제안 (최종) 예측: 메모리에 있으면 그대로, 없으면 저장된 CSV ----
    if 'PRED' in globals() and 'test' in PRED:
        P0 = PRED['test']
    else:
        _csv = pd.read_csv(f'{OUT2_DIR}/test_proposed_preds.csv')[['vid', 'tid', 'w', 'd', 'h']]
        P0 = ALL['test'][['vid', 'tid', 'mapped', 'w_gt', 'd_gt', 'h_gt']].merge(_csv, on=['vid', 'tid'], how='left')
        assert P0[['w', 'd', 'h']].notna().all().all(), 'test_proposed_preds.csv에 없는 트랙이 있음 → [F-2] 출력 확인'
        print(f'📄 제안 (최종): 저장된 test_proposed_preds.csv 사용 ({len(P0)}개 트랙)')
    key = list(zip(P0.vid, P0.tid.astype(int)))

    if 'ROW_TEST' not in globals():                       # [T-1]을 안 돌렸을 때: 같은 함수로 ② · ③만 계산
        _R = box_records(LABELS['test'], MAPS['test'], P0); _mt = _R[_R.kind == '매칭']
        ROW_TEST = {'② cm': _mt.cm.mean(), '② %': _mt.pct.mean(), '③ cm': _R.cm.mean(), '③ %': _R.pct.mean()}
    print(f"확인: 제안 (최종) test ② {ROW_TEST['② cm']:.2f} cm (보고서 값 1.75와 같아야 함)")

    # ---- baseline 예측 모으기 ----
    BASES = {}
    lr = lift_raw('test')
    if lr:
        BASES['OVMono3D-LIFT 원본'] = np.array([lr.get(k, [np.nan] * 3) for k in key], float)
        lrr = lift_raw('test', rail=True)
        BASES['OVMono3D-LIFT + 레일 보정'] = np.array([lrr.get(k, [np.nan] * 3) for k in key], float)
    for fname, label in (('test_lift_ft_A1.csv', 'OVMono3D-LIFT 미세조정 A-1'),
                         ('test_lift_ft_A1R.csv', 'OVMono3D-LIFT 미세조정 + 레일 A-1R')):
        em = extra_csv(fname)
        if em is None:
            print(f'ℹ️ {fname} 없음 → "{label}" 건너뜀'); continue
        BASES[label] = np.array([em.get(k, (np.nan,) * 3) for k in key], float)
        n_nan = int(np.isnan(BASES[label]).any(1).sum())
        if n_nan:
            print(f'⚠️ {label}: 예측 없는 트랙 {n_nan}개 (NaN) → 평균에서 빠질 수 있음')

    # ---- 채점 + bootstrap ----
    rows = {'제안 (최종)': {'② cm': ROW_TEST['② cm'], '② %': ROW_TEST['② %'], '③ cm': ROW_TEST['③ cm'], '③ %': ROW_TEST['③ %']}}
    m = P0.mapped.values; G = sort3(P0.loc[m, ['w_gt', 'd_gt', 'h_gt']].values)
    e0 = np.abs(sort3(P0.loc[m, ['w', 'd', 'h']].values) - G).mean(1); v = P0.vid.values[m]
    vids = np.unique(v); idx = {u: np.where(v == u)[0] for u in vids}
    for name, A in BASES.items():
        pred = P0[['vid', 'tid', 'mapped']].assign(w=A[:, 0], d=A[:, 1], h=A[:, 2])
        R_b = box_records(LABELS['test'], MAPS['test'], pred); mt = R_b[R_b.kind == '매칭']
        eb = np.abs(sort3(A[m]) - G).mean(1); rng = np.random.default_rng(0)
        bs = [(eb - e0)[np.concatenate([idx[u] for u in rng.choice(vids, len(vids))])].mean() for _ in range(2000)]
        rows[name] = {'② cm': mt.cm.mean(), '② %': mt.pct.mean(), '③ cm': R_b.cm.mean(), '③ %': R_b.pct.mean(),
                      'baseline − 최종 (② cm)': f'{(eb - e0).mean():+.2f} [{np.percentile(bs, 2.5):+.2f}, {np.percentile(bs, 97.5):+.2f}]'}
    hr('▶ TEST — 제안 (최종) vs OVMono3D-LIFT (같은 bbox · focal · 대표 프레임)')
    display(pd.DataFrame(rows).T.round(2))
    print('※ 확인: LIFT 원본 ② 8.68 · LIFT + 레일 보정 ② 6.81 · A-1 ② 3.45 cm (보고서 값과 같아야 함)')
    if not BASES:
        print('⚠️ baseline 예측 파일이 없음 → OVMono3D 노트북 · ablation 노트북의 출력 경로 확인')

📄 제안 (최종): 저장된 test_proposed_preds.csv 사용 (477개 트랙)
확인: 제안 (최종) test ② 1.75 cm (보고서 값 1.75와 같아야 함)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ TEST — 제안 (최종) vs OVMono3D-LIFT (같은 bbox · focal · 대표 프레임)


,② cm,② %,③ cm,③ %,baseline − 최종 (② cm)
제안 (최종),1.750553,8.909081,2.155411,11.732146,NaN
OVMono3D-LIFT 원본,8.678193,46.997441,8.984557,48.640082,"+6.93 [+6.05, +7.97]"
OVMono3D-LIFT + 레일 보정,6.814969,34.984244,7.241079,36.999195,"+5.06 [+4.40, +5.78]"
OVMono3D-LIFT 미세조정 A-1,3.448144,16.855361,3.855623,19.432158,"+1.70 [+1.53, +1.88]"
OVMono3D-LIFT 미세조정 + 레일 A-1R,3.441329,16.649432,3.853234,19.232611,"+1.69 [+1.49, +1.90]"


※ 확인: LIFT 원본 ② 8.68 · LIFT + 레일 보정 ② 6.81 · A-1 ② 3.45 cm (보고서 값과 같아야 함)


# **8. bin packing 입력 내보내기** — split마다 CSV 1개 (`gt` · `prop` = 최종 · `lift_raw` · `lift_ft_A1`, 세 변 큰 순)

In [ ]:
# [X-1] bin packing 입력 — split마다 CSV 1개
EXP_DIR = f'{PROJECT_DIR}/outputs/binpack'; EXTRA_DIR = f'{EXP_DIR}/extra'
os.makedirs(EXP_DIR, exist_ok=True); os.makedirs(EXTRA_DIR, exist_ok=True)


def lift_raw(sp):
    out = {}
    for p in glob.glob(f'{PROJECT_DIR}/outputs/baseline/ovmono3d_rep/raw_{sp}/*.pkl'):
        with open(p, 'rb') as f:
            r = pickle.load(f)
        for o in r.get('lift', []):
            out[(r['video_id'], int(o['tid']))] = np.asarray(o['dims_m'], float) * 100
    return out


for sp in SPLITS:
    P = PRED[sp]
    first = {(vid, int(t['tid'])): int(np.min(t['fi'])) * int(RECS[sp][vid]['step']) for vid in RECS[sp] for t in RECS[sp][vid]['tracks']}
    key = list(zip(P.vid, P.tid.astype(int)))
    sets = {'gt': P[['w_gt', 'd_gt', 'h_gt']].values, 'prop': P[['w', 'd', 'h']].values}
    lr = lift_raw(sp)
    if lr:
        sets['lift_raw'] = np.array([lr.get(k, [np.nan] * 3) for k in key])
    for p in sorted(glob.glob(f'{EXTRA_DIR}/{sp}_*.csv')):
        e = pd.read_csv(p); nm = os.path.basename(p)[len(sp) + 1:-4]
        if nm.startswith('prop'): continue                             # 예전 확장 모델 파일 (지금은 prop = 최종)
        em = {(v, int(t)): (a, b, c) for v, t, a, b, c in e[['vid', 'tid', 'w', 'd', 'h']].itertuples(index=False)}
        sets[nm] = np.array([em.get(k, (np.nan,) * 3) for k in key], float)
    out = pd.DataFrame({'split': sp, 'vid': P.vid.values, 'tid': P.tid.values.astype(int), 'box_id': P.box_id.values,
                        'mapped': P.mapped.values, 'n_boxes_in_video': P.n_gt.values, 'first_frame': [first[k] for k in key],
                        'focal_mm': P.focal_mm.values, 'sin_t': P.sin_t.values, 'cos_t': P.cos_t.values})
    for name, arr in sets.items():
        s = sort3(arr)
        for j in range(3):
            out[f'{name}_{j + 1}'] = s[:, j]
    path = f'{EXP_DIR}/{sp}_binpack_input.csv'; out.to_csv(path, index=False)
    print(f'{sp}: {len(out)}개 트랙 (매칭 {int(out.mapped.sum())}) · 열 {list(sets)} → {path}')

train: 910개 트랙 (매칭 899) · 열 ['gt', 'prop', 'lift_raw', 'lift_ft_A1'] → /content/drive/MyDrive/4학년 1학기 과목별 정리/소프트웨어융합캡스톤디자인/BoxSizing/outputs/binpack/train_binpack_input.csv
val: 467개 트랙 (매칭 461) · 열 ['gt', 'prop', 'lift_raw', 'lift_ft_A1'] → /content/drive/MyDrive/4학년 1학기 과목별 정리/소프트웨어융합캡스톤디자인/BoxSizing/outputs/binpack/val_binpack_input.csv
test: 477개 트랙 (매칭 469) · 열 ['gt', 'prop', 'lift_raw', 'lift_ft_A1'] → /content/drive/MyDrive/4학년 1학기 과목별 정리/소프트웨어융합캡스톤디자인/BoxSizing/outputs/binpack/test_binpack_input.csv


# **9. 런타임 — 배치 처리 효과 · 박스당 추론 시간 · 실시간 여부**
- **환경:** [R-0]에 GPU · CUDA · PyTorch · CPU · RAM을 기록한다 (보고서에 그대로 적기). A100 기준으로 설계했지만 다른 GPU에서도 돈다
- **[R-1] 배치 크기:** 같은 crop 128장을 batch 1 ~ 128로 처리해 crop당 시간이 더 줄지 않는 크기(GPU 포화)를 찾는다 (DINOv3 · DA3 · 둘 다)
- **[R-2] 치수 추정만 (트래킹 캐시가 있다고 가정):** test 영상 10개를 세 방식으로 처리한다
  - **기존:** 프레임마다 위치 이동(seek) 읽기 + crop 한 장씩 (배치 없음)
  - **배치:** 처음부터 순서대로 읽기(같은 프레임은 한 번) + 영상의 모든 crop을 한 번에 (원본 · 반전 함께)
  - **스트리밍:** 프레임은 이미 메모리에 있음 → 트랙이 끝날 때마다 그 트랙 crop만 한 배치로 → **박스당 지연**
- **[R-3] 전체 파이프라인:** 영상 → ROI → YOLO26-L v5 검출 → 추적(Kalman 2회 + 병합) → 레일 seg → 프레임 선택 · crop → DA3 + DINOv3 → head를 같은 영상 10개로 처음부터 끝까지 실행한다
  - 추적 · ROI · 레일 RANSAC은 캐시를 만든 `main.py` 코드를 그대로 가져오고 (`PROJECT_DIR/main.py`에 두기), 검출 · 레일 seg는 같은 가중치의 PyTorch(`.pt`)로 돌린다
  - 먼저 캐시와 같은 트랙 · 레일이 나오는지 확인한 뒤 단계별 시간을 낸다. YOLO는 한 장씩(main.py와 같음) · 16장씩 배치 두 방식
  - `main.py` 추적은 영상 전체를 본 뒤 결과가 나오므로 **처리량 기준 실시간**(영상 처리 시간 ÷ 영상 길이)으로 판단한다
- **[R-4] 요약:** 치수 추정만 · 전체 파이프라인의 박스당 시간 · 실시간 배율(평균 · 가장 느린 영상) · ✅ / ✗

In [ ]:
# [R-0] 실행 환경 기록 — 런타임 결과는 이 환경 기준 (보고서에 그대로 적기)
#  - RUN_RUNTIME = False (기본): 이미 측정해 저장한 결과(OUT2_DIR/runtime_summary.json)를 불러와 표만 다시 보여 준다
#    True: [R-1] ~ [R-3]을 다시 측정하고 json을 새로 저장한다 (A100에서 수십 분)
RUN_RUNTIME = True  #@param {type:"boolean"}
import platform
try:
    import psutil; _ram = f'{psutil.virtual_memory().total / 2**30:.0f} GB'
except Exception:
    _ram = '?'
_cpu = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), platform.processor())
_gp = torch.cuda.get_device_properties(0)
ENV = {'GPU': _gp.name, 'GPU 메모리': f'{_gp.total_memory / 2**30:.0f} GB', 'CUDA (torch)': torch.version.cuda,
       'cuDNN': torch.backends.cudnn.version(), 'PyTorch': torch.__version__, 'timm': timm.__version__, 'OpenCV': cv2.__version__,
       'Python': platform.python_version(), 'CPU': f'{_cpu} × {os.cpu_count()} 스레드', 'RAM': _ram,
       '정밀도': 'fp16 autocast (torch.autocast)', '입력': f'crop 224 → {TOK_RES} (DINOv3 patch 16 · DA3 patch 14)',
       '모델': 'DA3 인코더 304.4 M + DINOv3-L 303.1 M (고정) + head 10개 (STEP 1 · 2 × fold 5)',
       '추론 설정': '트랙당 최대 8장 · 좌우반전 TTA (원본 + 반전) · fold 5개 평균'}
hr('▶ 실행 환경'); display(pd.Series(ENV, name='값').to_frame())
if 'A100' not in _gp.name:
    print(f'⚠️ A100이 아님 ({_gp.name}) → 결과 해석 시 GPU 이름을 함께 적을 것')
torch.backends.cudnn.benchmark = True
RUNTIME_JSON = f'{OUT2_DIR}/runtime_summary.json'
RJ = json.load(open(RUNTIME_JSON)) if os.path.exists(RUNTIME_JSON) else {}
if not RUN_RUNTIME:
    if RJ.get('per_video'):
        print('⏭ 저장된 런타임 결과를 씀 — 아래 표의 값은 이 측정 환경 기준 (지금 GPU와 다를 수 있음)')
        display(pd.Series(RJ['env'], name='측정 환경').to_frame()); ENV = RJ['env']
    else:
        print('⚠️ 저장된 결과가 없거나 일부만 있음 → 이번에는 측정함 (RUN_RUNTIME = True)'); RUN_RUNTIME = True


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ 실행 환경


,값
GPU,NVIDIA A100-SXM4-40GB
GPU 메모리,39 GB
CUDA (torch),13.0
cuDNN,92700
PyTorch,2.11.0+cu130
timm,1.0.30
OpenCV,4.14.0
Python,3.13.16
CPU,Intel(R) Xeon(R) CPU @ 2.20GHz × 12 스레드
RAM,83 GB


In [ ]:
# [R-1] 배치 크기별 crop당 시간 — DINOv3 · DA3 · 둘 다 (최종), TTA 없음 · 448 · fp16
#  - 같은 crop 128장을 batch 1 · 8 · 16 · 32 · 64 · 128로 나눠 처리, warmup 후 torch.cuda.synchronize · 반복 중앙값
#  - L4처럼 작은 GPU는 batch 1로도 꽉 차서 효과가 작고, A100은 배치로 묶을수록 빨라짐
import statistics
R_REP = 5


def _sync():
    if DEV == 'cuda': torch.cuda.synchronize()


def timeit(fn, rep=R_REP, warm=2):
    for _ in range(warm): fn()
    ts = []
    for _ in range(rep):
        _sync(); t0 = time.perf_counter(); fn(); _sync(); ts.append(time.perf_counter() - t0)
    return statistics.median(ts)


# 모델 (고정) — DINOv3-L · DA3 인코더
def load_v3():
    global V3, V3_M, V3_S, V3_NP, V3_G
    V3 = timm.create_model(DINO_NAME, pretrained=True, num_classes=0, img_size=TOK_RES).to(DEV).eval()
    _dc = timm.data.resolve_model_data_config(V3)
    V3_M, V3_S = (torch.tensor(_dc[k], device=DEV).view(1, 3, 1, 1) for k in ('mean', 'std'))
    V3_NP, V3_G = int(V3.num_prefix_tokens), TOK_RES // int(V3.patch_embed.patch_size[0])


@torch.no_grad()
def v3_tokens_gpu(x01):
    with torch.autocast('cuda', dtype=torch.float16, enabled=DEV == 'cuda'):
        h = V3.forward_features((x01 - V3_M) / V3_S)
    gr = h[:, V3_NP:].float().transpose(1, 2).reshape(len(h), -1, V3_G, V3_G)
    return F.adaptive_avg_pool2d(gr, GRID).flatten(2).transpose(1, 2)


ENC = {'DINOv3-L': [v3_tokens_gpu], 'DA3 인코더': [lambda x: da3_tokens_gpu(DA3_ENC, x)],
       'DA3 + DINOv3 (최종)': [lambda x: da3_tokens_gpu(DA3_ENC, x), v3_tokens_gpu]}


@torch.no_grad()
def encode(crops_u8, encs, bs, tta=False):
    """uint8 crop (N, 224, 224, 3) → 토큰 (N, 49, D) GPU. tta면 원본 · 반전을 한 배치에 함께 넣음 → (2N, 49, D)"""
    X = torch.from_numpy(crops_u8).to(DEV).permute(0, 3, 1, 2).float() / 255
    if tta: X = torch.cat([X, X.flip(3)])
    out = []
    for i in range(0, len(X), bs):
        x = F.interpolate(X[i:i + bs], size=(TOK_RES, TOK_RES), mode='bilinear', align_corners=False)
        out.append(torch.cat([fn(x) for fn in encs], -1))
    return torch.cat(out)


if RUN_RUNTIME or 'batch_table' not in RJ:                            # 측정 (저장 결과에 이 표가 없으면 이것만 1 ~ 2분 측정)
    load_v3(); DA3_ENC = da3_encoder()
    _crops = FC[('val', 'mapped')][:128]
    rows = []
    for name, encs in ENC.items():
        for bs in (1, 8, 16, 32, 64, 128):
            t = timeit(lambda: encode(_crops, encs, bs))
            rows.append({'모델': name, 'batch': bs, 'crop당 ms': t / len(_crops) * 1000})
    TB = pd.DataFrame(rows).pivot(index='batch', columns='모델', values='crop당 ms')[list(ENC)]
    RJ['batch_table'] = TB.reset_index().to_dict('records'); RJ['best_batch'] = int(TB['DA3 + DINOv3 (최종)'].idxmin())
    if not RUN_RUNTIME and RJ.get('per_video'):
        json.dump(RJ, open(RUNTIME_JSON, 'w'), ensure_ascii=False, indent=1, default=float)
else:
    TB = pd.DataFrame(RJ['batch_table']).set_index('batch')
hr(f'▶ [R-1] 배치 크기별 crop당 시간 ({ENV["GPU"]}, 448, fp16, TTA 없음)')
display(TB.round(2))
BEST_BS = int(RJ.get('best_batch', TB['DA3 + DINOv3 (최종)'].idxmin()))
print(f'→ 최종 모델의 가장 빠른 batch: {BEST_BS} · batch 1 대비 {TB.loc[1, "DA3 + DINOv3 (최종)"] / TB.loc[BEST_BS, "DA3 + DINOv3 (최종)"]:.1f}배 빠름')

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

  from pkg_resources import resource_stream, resource_exists

Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

[INFO ] using MLP layer as FFN


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ [R-1] 배치 크기별 crop당 시간 (NVIDIA A100-SXM4-40GB, 448, fp16, TTA 없음)


모델,DINOv3-L,DA3 인코더,DA3 + DINOv3 (최종)
batch,,,
1,27.20,22.30,49.03
8,6.39,7.28,13.67
16,5.99,6.19,12.18
32,5.71,5.60,11.34
64,5.61,5.33,10.96
128,5.77,5.20,10.98


→ 최종 모델의 가장 빠른 batch: 64 · batch 1 대비 4.5배 빠름


In [ ]:
# [R-2] 박스당 추론 시간 · 배치 처리 효과 · 실시간 여부 — 최종 모델 (DA3 + DINOv3, 트랙당 최대 8장, TTA)
#  - 대상: test 영상 R_N개 (영상 길이 · 상자 수 · 프레임 수가 다양한 순서대로)
#  - 단계: ① 영상에서 필요한 프레임 읽기 + crop  ② DA3 + DINOv3 토큰  ③ head 10개 (STEP 1 · 2 × fold 5)
#  - 비교하는 세 가지 처리 방식
#      기존     : 프레임마다 위치 이동(seek) 읽기 + crop 한 장씩(batch 1) 원본 · 반전 따로
#      배치     : 처음부터 순서대로 읽기(같은 프레임은 한 번) + 영상의 모든 crop을 한 번에 (원본 · 반전 함께, batch BEST_BS)
#      스트리밍 : 현장 가정 — 프레임은 이미 메모리에 있음 (읽기 0), 트랙이 끝날 때마다 그 트랙 crop만 한 배치로
#  - 실시간 기준: 처리 시간 ÷ 영상 길이 = 실시간 배율 < 1, 그리고 박스당 지연 < 평균 상자 간격
#  - 여기서는 치수 추정 부분만 (트래킹 캐시가 있다고 가정). YOLO · 추적 · 레일까지 포함한 전체 파이프라인은 [R-3]
R_N = 10
HEADS = []
if RUN_RUNTIME:
    for cfg in (STEP1, STEP2):
        for k in range(N_FOLD):
            ck = torch.load(f"{CKPT2}/{cfg['name']}_fold{k}.pt", map_location='cpu', weights_only=False)
            h = TokHead(np.zeros(len(cfg['tgt'])), len(cfg['side']), 49, 2048).to(DEV).eval(); h.load_state_dict(ck['state'])
            HEADS.append((h, len(cfg['side'])))


@torch.no_grad()
def run_heads(tok):
    """tok: (2N, 49, 2048) = 원본 N + 반전 N (TTA) 또는 (N, ...) → head 10개"""
    for h, ns in HEADS:
        h(tok, torch.zeros(len(tok), ns, device=DEV))


def read_frames(sp, vid, need, mode):
    frames, cap = {}, cv2.VideoCapture(video_path(sp, vid))
    if mode == 'seek':
        for f in need:
            cap.set(cv2.CAP_PROP_POS_FRAMES, f); ok_, fr = cap.read()
            if ok_: frames[f] = fr
    else:
        want, i = set(need), 0
        while i <= need[-1]:
            if not cap.grab(): break
            if i in want: _, frames[i] = cap.retrieve()
            i += 1
    cap.release()
    return frames


def crops_from(frames, f):
    imgs = np.zeros((len(f), IMG, IMG, 3), np.uint8)
    for j, r in enumerate(f.itertuples()):
        fr = frames.get(int(r.rep_fi) * int(r.step))
        if fr is None: continue
        H_, W_ = fr.shape[:2]; px, py = r.bw * PAD, r.bh * PAD
        c = fr[int(max(0, r.y1 - py)):int(min(H_, r.y2 + py)), int(max(0, r.x1 - px)):int(min(W_, r.x2 + px))]
        imgs[j] = letterbox(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
    return imgs


def video_len_s(sp, vid):
    cap = cv2.VideoCapture(video_path(sp, vid)); n, fps = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS); cap.release()
    return n / fps, fps


ENC_F = ENC['DA3 + DINOv3 (최종)']
FRV = pd.concat([FR[('test', p)] for p in ('mapped', 'unmapped') if FR[('test', p)] is not None]).reset_index(drop=True)
_nb = FRV.groupby('vid').tid.nunique().sort_values()
R_VIDS = list(_nb.index[np.linspace(0, len(_nb) - 1, R_N).round().astype(int)])     # 상자 수가 적은 영상 ~ 많은 영상 고르게
if RUN_RUNTIME:
    rows = []
    for vid in R_VIDS:
        f = FRV[FRV.vid == vid].reset_index(drop=True); n_trk = f.tid.nunique()
        dur, fps = video_len_s('test', vid)
        need = sorted({int(fi) * int(st) for fi, st in zip(f.rep_fi, f.step)})
        fr_all = read_frames('test', vid, need, 'seq'); crops = crops_from(fr_all, f)
        by_trk = [crops[np.where(f.tid.values == t)[0]] for t in f.tid.unique()]

        t_read_seek = timeit(lambda: crops_from(read_frames('test', vid, need, 'seek'), f), rep=2, warm=1)
        t_read_seq = timeit(lambda: crops_from(read_frames('test', vid, need, 'seq'), f), rep=2, warm=1)

        def old():                                                            # 한 장씩 · 원본 · 반전 따로
            for c in crops:
                for fl in (False, True):
                    x = torch.from_numpy(c[None]).to(DEV).permute(0, 3, 1, 2).float() / 255
                    if fl: x = x.flip(3)
                    x = F.interpolate(x, size=(TOK_RES, TOK_RES), mode='bilinear', align_corners=False)
                    run_heads(torch.cat([fn(x) for fn in ENC_F], -1))
        t_model_old = timeit(old, rep=2, warm=1)
        t_model_batch = timeit(lambda: run_heads(encode(crops, ENC_F, BEST_BS, tta=True)), rep=3)
        t_trk = [timeit(lambda c=c: run_heads(encode(c, ENC_F, BEST_BS, tta=True)), rep=3) for c in by_trk]
        rows.append({'vid': vid, '영상 길이 s': dur, 'fps': fps, '트랙': n_trk, 'crop': len(f), '읽은 프레임': len(need),
                     '평균 상자 간격 s': dur / n_trk,
                     '기존 ms/박스': (t_read_seek + t_model_old) / n_trk * 1000,
                     '배치 ms/박스': (t_read_seq + t_model_batch) / n_trk * 1000,
                     '스트리밍 지연 ms/박스 (평균)': np.mean(t_trk) * 1000, '스트리밍 지연 ms/박스 (최대)': np.max(t_trk) * 1000,
                     '읽기 seek ms/박스': t_read_seek / n_trk * 1000, '읽기 순차 ms/박스': t_read_seq / n_trk * 1000,
                     '모델 한 장씩 ms/박스': t_model_old / n_trk * 1000, '모델 배치 ms/박스': t_model_batch / n_trk * 1000,
                     '실시간 배율 기존': (t_read_seek + t_model_old) / dur, '실시간 배율 배치': (t_read_seq + t_model_batch) / dur,
                     '실시간 배율 스트리밍': np.sum(t_trk) / dur})
        print(f'  {vid}: {dur:.1f}s · 트랙 {n_trk} · crop {len(f)} 완료')
    RT = pd.DataFrame(rows).set_index('vid')
else:
    RT = pd.DataFrame(RJ['per_video']).set_index('vid')
hr(f'▶ [R-2] 영상별 결과 ({ENV["GPU"]})'); display(RT.round(3))

  test_025: 5.3s · 트랙 2 · crop 16 완료
  test_012: 5.3s · 트랙 5 · crop 40 완료
  test_008: 5.9s · 트랙 6 · crop 48 완료
  test_030: 8.9s · 트랙 8 · crop 61 완료
  test_041: 10.1s · 트랙 9 · crop 70 완료
  test_022: 9.5s · 트랙 10 · crop 79 완료
  test_005: 14.3s · 트랙 11 · crop 88 완료
  test_026: 12.5s · 트랙 12 · crop 95 완료
  test_004: 10.7s · 트랙 15 · crop 109 완료
  test_038: 15.5s · 트랙 20 · crop 160 완료

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ [R-2] 영상별 결과 (NVIDIA A100-SXM4-40GB)


,영상 길이 s,fps,트랙,crop,읽은 프레임,평균 상자 간격 s,기존 ms/박스,배치 ms/박스,스트리밍 지연 ms/박스 (평균),스트리밍 지연 ms/박스 (최대),읽기 seek ms/박스,읽기 순차 ms/박스,모델 한 장씩 ms/박스,모델 배치 ms/박스,실시간 배율 기존,실시간 배율 배치,실시간 배율 스트리밍
vid,,,,,,,,,,,,,,,,,
test_025,5.267,30.190,2,16,16,2.633,1139.311,221.243,195.447,195.549,308.579,40.337,830.731,180.906,0.433,0.084,0.074
test_012,5.267,30.190,5,40,31,1.053,1065.207,200.854,195.429,195.604,230.560,22.520,834.647,178.333,1.011,0.191,0.186
test_008,5.867,30.170,6,48,36,0.978,1062.465,193.769,195.659,196.039,231.608,16.991,830.857,176.778,1.087,0.198,0.200
test_030,8.867,30.113,8,61,45,1.108,1092.593,183.579,187.579,195.767,300.183,17.251,792.411,166.328,0.986,0.166,0.169
test_041,10.067,30.099,9,70,52,1.119,1105.664,191.355,190.952,195.925,297.074,19.615,808.589,171.740,0.989,0.171,0.171
test_022,9.467,30.106,10,79,46,0.947,1098.798,190.707,193.726,196.019,273.399,17.270,825.399,173.437,1.161,0.201,0.205
test_005,14.267,30.070,11,88,70,1.297,1163.614,193.824,195.798,196.164,333.079,18.886,830.534,174.938,0.897,0.149,0.151
test_026,12.467,30.080,12,95,64,1.039,1072.060,187.708,193.956,196.320,247.414,15.184,824.646,172.524,1.032,0.181,0.187
test_004,10.667,30.094,15,109,70,0.711,972.323,172.506,180.072,196.145,215.113,13.136,757.210,159.370,1.367,0.243,0.253


In [ ]:
# [R-3a] 트래킹 캐시를 만든 추적 · ROI · 레일 계산 코드 (main.py와 같은 코드, 노트북에 그대로 작성)
#  - 상수 34개 · 함수 17개: 움직임 기반 ROI · 0.1초 간격 프레임 추출 · Kalman 추적 2회 · 트랙 병합 · 레일 RANSAC
#  - 검출(YOLO) · 레일 세그멘테이션은 [R-3]에서 같은 가중치의 PyTorch(.pt)로 실행
#  - 아래 함수 · 상수는 TRK.이름 으로도 쓸 수 있게 묶어 둔다 ([R-3] · [V-0]이 TRK.xxx로 부름)
import types
try:
    import filterpy  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'filterpy'], check=True)

import numpy as np
import cv2
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import RANSACRegressor
from collections import defaultdict

# ---- 상수 ----
MOTION_THRESHOLD    = 44
MOTION_RATIO_THRESH = 0.0294
ROI_PAD             = 20
BASE_MAX_W_RATIO     = 0.70
BASE_MAX_H_RATIO     = 0.70
RELAXED_MAX_W_RATIO  = 0.90
RELAXED_MAX_H_RATIO  = 0.90
BASE_MIN_CONF     = 0.6
RELAXED_MIN_CONF  = 0.85
DUPLICATE_OVERLAP_THRESH = 0.90
LABEL_INTERVAL_SEC  = 0.1
CONF_THRESH         = 0.25
IOU_THRESH          = 0.45
BT_HIGH_THRESH  = 0.50
BT_LOW_THRESH   = 0.25
BT_IOU_THRESH_1 = 0.30
BT_IOU_THRESH_2 = 0.20
BT_MAX_MISS     = 45
MERGE_IOU_THRESH    = 0.20
MERGE_MAX_FRAME_GAP = 30
POST_NMS_THRESH = 0.60
LANE_TOLERANCE_RATIO = 0.6
DEPTH_DIST_RATIO     = 1.5
FORWARD_ONLY_TOLERANCE_RATIO = 0.3
PARTIAL_AREA_RATIO         = 0.5
EDGE_TOLERANCE_RATIO       = 0.35
CROSS_AXIS_TOLERANCE_RATIO = 0.7
MIN_TRACK_FRAMES_FOR_MERGE = 2
MIN_TRACK_FRAMES_FOR_COUNT = 2
GLOBAL_MOTION_THRESH = MOTION_THRESHOLD
GLOBAL_RATIO_THRESH  = MOTION_RATIO_THRESH
RAIL_N_SAMPLE = 30
RAIL_MIN_OK = 3

# ---- 함수 ----

def post_nms(dets, iou_thresh=POST_NMS_THRESH):
    if len(dets) == 0:
        return dets
    dets_sorted = sorted(dets, key=lambda d: d[4], reverse=True)
    keep = []
    while dets_sorted:
        best = dets_sorted.pop(0)
        keep.append(best)
        remaining = []
        for d in dets_sorted:
            ax1, ay1, ax2, ay2 = best[:4]
            bx1, by1, bx2, by2 = d[:4]
            ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
            ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
            inter = max(0, ix2-ix1) * max(0, iy2-iy1)
            if inter == 0:
                remaining.append(d); continue
            ua = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter
            if inter / ua < iou_thresh:
                remaining.append(d)
        dets_sorted = remaining
    return keep


def extract_frames(video_path, interval_sec=LABEL_INTERVAL_SEC):
    cap  = cv2.VideoCapture(video_path)
    fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(int(round(fps * interval_sec)), 1)
    out, idx = [], 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if idx % step == 0:
            out.append((frame, idx / fps))
        idx += 1
    cap.release()
    return out, fps


def get_roi_for_test_video(video_path, motion_threshold=GLOBAL_MOTION_THRESH,
                            motion_ratio_thresh=GLOBAL_RATIO_THRESH):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    min_sample_frames = 20
    frame_step = max(1, min(int(fps * 0.5), max(total_f // min_sample_frames, 1)))
    extra = set(range(min(5, total_f))) | set(range(max(0, total_f - 5), total_f))
    sample_indices = np.array(sorted(set(np.arange(0, total_f, frame_step, dtype=int).tolist()) | extra))
    sampled_frames = []
    for idx in sample_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            sampled_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()
    if len(sampled_frames) < 3:
        return None
    stack = np.stack(sampled_frames, axis=0).astype(np.float32)
    bg_median = np.median(stack, axis=0)
    motion_count = np.zeros((orig_h, orig_w), dtype=np.float32)
    for frame_gray in sampled_frames:
        diff = np.abs(frame_gray.astype(np.float32) - bg_median)
        motion_count += (diff > motion_threshold).astype(np.float32)
    motion_ratio = motion_count / len(sampled_frames)
    duration_sec = total_f / max(fps, 1)
    adaptive_thresh = motion_ratio_thresh
    if duration_sec < 5: adaptive_thresh = motion_ratio_thresh * 0.4
    elif duration_sec < 10: adaptive_thresh = motion_ratio_thresh * 0.7
    motion_mask = (motion_ratio >= adaptive_thresh).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_CLOSE, kernel)
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(motion_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return (0, 0, orig_w, orig_h)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    bx, by, bw, bh = cv2.boundingRect(contours[0])
    pad = 20
    return (max(0, bx-pad), max(0, by-pad), min(orig_w, bx+bw+pad), min(orig_h, by+bh+pad))


def estimate_conveyor_speed(all_tracks):
    vx_list, vy_list = [], []
    for tid, hist in all_tracks.items():
        if len(hist) < 3: continue
        frames = [h[0] for h in hist]
        bboxes = [h[1] for h in hist]
        for k in range(1, len(frames)):
            dt = frames[k] - frames[k-1]
            if dt <= 0 or dt > 5: continue
            cx_prev = (bboxes[k-1][0] + bboxes[k-1][2]) / 2
            cy_prev = (bboxes[k-1][1] + bboxes[k-1][3]) / 2
            cx_cur  = (bboxes[k][0]   + bboxes[k][2])   / 2
            cy_cur  = (bboxes[k][1]   + bboxes[k][3])   / 2
            vx_list.append((cx_cur - cx_prev) / dt)
            vy_list.append((cy_cur - cy_prev) / dt)
    if not vx_list:
        return 0.0, 0.0
    return float(np.median(vx_list)), float(np.median(vy_list))


def compute_dynamic_params(vx, vy, frame_interval_sec=LABEL_INTERVAL_SEC, bbox_avg_px=150):
    speed = np.sqrt(vx**2 + vy**2)
    if speed < 1.0:
        return 30, 35
    frames_per_box = bbox_avg_px / speed
    max_miss  = int(np.clip(frames_per_box * 1.5, 5, BT_MAX_MISS))
    merge_gap = int(np.clip(frames_per_box * 2.0, 5, MERGE_MAX_FRAME_GAP))
    return int(max_miss), int(merge_gap)


def build_spatial_order(all_tracks, conveyor_vx=0.0, conveyor_vy=0.0):
    track_profiles = {}
    for tid, hist in all_tracks.items():
        if not hist: continue
        frames = [h[0] for h in hist]
        bboxes = [h[1] for h in hist]
        cx_vals = [(b[0]+b[2])/2 for b in bboxes]
        cy_vals = [(b[1]+b[3])/2 for b in bboxes]
        w_vals = [b[2]-b[0] for b in bboxes]
        h_vals = [b[3]-b[1] for b in bboxes]
        cx_rep = float(np.median(cx_vals))
        first_idx = int(np.argmin(frames))
        cy_entry = cy_vals[first_idx]
        track_profiles[tid] = {
            'cx': cx_rep, 'cy_entry': cy_entry,
            'first_frame': min(frames), 'last_frame': max(frames),
            'avg_w': float(np.median(w_vals)), 'avg_h': float(np.median(h_vals)),
        }
    return track_profiles


def is_same_lane(prof_a, prof_b, lane_tolerance_ratio=LANE_TOLERANCE_RATIO):
    avg_w = (prof_a['avg_w'] + prof_b['avg_w']) / 2
    cx_diff = abs(prof_a['cx'] - prof_b['cx'])
    return cx_diff < avg_w * lane_tolerance_ratio


def depth_order_consistent(earlier_prof, later_prof, conveyor_vy=0.0):
    if abs(conveyor_vy) < 0.5:
        return True
    cy_diff = later_prof['cy_entry'] - earlier_prof['cy_entry']
    if conveyor_vy > 0:
        tolerance = (earlier_prof['avg_h'] + later_prof['avg_h']) / 2
        return cy_diff > -tolerance
    else:
        tolerance = (earlier_prof['avg_h'] + later_prof['avg_h']) / 2
        return cy_diff < tolerance


class KalmanBoxTracker:
    count = 0
    def __init__(self, bbox, conf, init_vx=0.0, init_vy=0.0):
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([
            [1,0,0,0,1,0,0],[0,1,0,0,0,1,0],[0,0,1,0,0,0,1],[0,0,0,1,0,0,0],
            [0,0,0,0,1,0,0],[0,0,0,0,0,1,0],[0,0,0,0,0,0,1],
        ], dtype=float)
        self.kf.H = np.array([
            [1,0,0,0,0,0,0],[0,1,0,0,0,0,0],[0,0,1,0,0,0,0],[0,0,0,1,0,0,0],
        ], dtype=float)
        self.kf.R[2:, 2:] *= 10.0
        self.kf.P[4:, 4:] *= 1000.0
        self.kf.P *= 10.0
        self.kf.Q[-1,-1] *= 0.01
        self.kf.Q[4:,4:] *= 0.01
        self.kf.x[:4] = self._bbox_to_z(bbox)
        self.kf.x[4, 0] = init_vx
        self.kf.x[5, 0] = init_vy
        self.id = KalmanBoxTracker.count
        KalmanBoxTracker.count += 1
        self.hits = 1; self.miss = 0; self.conf = conf; self.history = []
        self.last_matched_bbox = tuple(map(int, bbox))

    def _bbox_to_z(self, bbox):
        x1, y1, x2, y2 = bbox
        cx = (x1 + x2) / 2.0; cy = (y1 + y2) / 2.0
        area = (x2 - x1) * (y2 - y1)
        ratio = (x2 - x1) / float(y2 - y1 + 1e-6)
        return np.array([[cx],[cy],[area],[ratio]], dtype=float)

    def _z_to_bbox(self):
        cx, cy, area, ratio = (self.kf.x[0,0], self.kf.x[1,0], self.kf.x[2,0], self.kf.x[3,0])
        area = max(area, 1.0); ratio = max(ratio, 0.1)
        w = np.sqrt(area * ratio); h = area / w
        return np.array([cx-w/2, cy-h/2, cx+w/2, cy+h/2])

    def predict(self):
        if self.kf.x[2,0] + self.kf.x[6,0] <= 0:
            self.kf.x[6,0] = 0.0
        self.kf.predict(); self.miss += 1
        return self._z_to_bbox()

    def update(self, bbox, conf, frame_idx):
        self.kf.update(self._bbox_to_z(bbox))
        self.hits += 1; self.miss = 0; self.conf = conf
        self.history.append((frame_idx, tuple(map(int, bbox)), conf))
        self.last_matched_bbox = tuple(map(int, bbox))

    def get_bbox(self):
        return self._z_to_bbox()


def _iou_matrix(trackers, detections):
    iou_mat = np.zeros((len(trackers), len(detections)), dtype=float)
    for i, t in enumerate(trackers):
        tb = t.get_bbox()
        for j, d in enumerate(detections):
            db = d[:4]
            xx1 = max(tb[0], db[0]); yy1 = max(tb[1], db[1])
            xx2 = min(tb[2], db[2]); yy2 = min(tb[3], db[3])
            inter = max(0, xx2-xx1) * max(0, yy2-yy1)
            if inter == 0: continue
            ua = ((tb[2]-tb[0])*(tb[3]-tb[1]) + (db[2]-db[0])*(db[3]-db[1]) - inter)
            iou_mat[i,j] = inter / ua
    return iou_mat


def _candidate_overlap_ratio(base_box, candidate_box):
    ax1, ay1, ax2, ay2 = base_box[:4]
    bx1, by1, bx2, by2 = candidate_box[:4]
    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0: return 0.0
    candidate_area = (bx2-bx1) * (by2-by1)
    if candidate_area <= 0: return 0.0
    return inter / candidate_area


def _hungarian_match(iou_mat, thresh):
    if iou_mat.size == 0:
        return [], list(range(iou_mat.shape[0])), list(range(iou_mat.shape[1]))
    row_ind, col_ind = linear_sum_assignment(-iou_mat)
    matched, unmatched_t, unmatched_d = [], [], []
    matched_rows, matched_cols = set(), set()
    for r, c in zip(row_ind, col_ind):
        if iou_mat[r,c] >= thresh:
            matched.append((r,c)); matched_rows.add(r); matched_cols.add(c)
    for r in range(iou_mat.shape[0]):
        if r not in matched_rows: unmatched_t.append(r)
    for c in range(iou_mat.shape[1]):
        if c not in matched_cols: unmatched_d.append(c)
    return matched, unmatched_t, unmatched_d


def _lane_distance_matrix(trackers, detections, lane_tolerance_ratio=LANE_TOLERANCE_RATIO):
    lane_mat = np.full((len(trackers), len(detections)), np.inf)
    for i, t in enumerate(trackers):
        tb = t.get_bbox(); tcx = (tb[0] + tb[2]) / 2; t_w = tb[2] - tb[0]
        for j, d in enumerate(detections):
            dcx = (d[0] + d[2]) / 2; d_w = d[2] - d[0]
            avg_w = (t_w + d_w) / 2; cx_diff = abs(tcx - dcx)
            if cx_diff < avg_w * lane_tolerance_ratio:
                tcy = (tb[1] + tb[3]) / 2; dcy = (d[1] + d[3]) / 2
                lane_mat[i, j] = abs(tcy - dcy) / max(avg_w, 1)
    return lane_mat


def _apply_forward_only_mask(trackers, detections, vx, vy, tolerance_ratio=FORWARD_ONLY_TOLERANCE_RATIO):
    n_t, n_d = len(trackers), len(detections)
    mask = np.ones((n_t, n_d), dtype=bool)
    speed = np.sqrt(vx**2 + vy**2)
    if speed < 0.5: return mask
    ux, uy = vx / speed, vy / speed
    for i, t in enumerate(trackers):
        lx1, ly1, lx2, ly2 = t.last_matched_bbox
        lcx, lcy = (lx1+lx2)/2, (ly1+ly2)/2
        lw, lh = (lx2-lx1), (ly2-ly1)
        for j, d in enumerate(detections):
            dx1, dy1, dx2, dy2 = d[:4]
            dcx, dcy = (dx1+dx2)/2, (dy1+dy2)/2
            dw, dh = (dx2-dx1), (dy2-dy1)
            disp_x, disp_y = dcx - lcx, dcy - lcy
            projection = disp_x * ux + disp_y * uy
            avg_size = (lw + lh + dw + dh) / 4
            tolerance = avg_size * tolerance_ratio
            if projection < -tolerance:
                mask[i, j] = False
    return mask


class ByteTracker:
    def __init__(self, high_thresh=BT_HIGH_THRESH, low_thresh=BT_LOW_THRESH,
                 iou_thresh_1=BT_IOU_THRESH_1, iou_thresh_2=BT_IOU_THRESH_2,
                 max_miss=BT_MAX_MISS, conveyor_vx=0.0, conveyor_vy=0.0,
                 pos_dist_thresh=DEPTH_DIST_RATIO, lane_tol_ratio=LANE_TOLERANCE_RATIO,
                 forward_only_tol=FORWARD_ONLY_TOLERANCE_RATIO,
                 duplicate_overlap_thresh=DUPLICATE_OVERLAP_THRESH):
        self.high_thresh = high_thresh; self.low_thresh = low_thresh
        self.iou_thresh_1 = iou_thresh_1; self.iou_thresh_2 = iou_thresh_2
        self.max_miss = max_miss; self.conveyor_vx = conveyor_vx; self.conveyor_vy = conveyor_vy
        self.pos_dist_thresh = pos_dist_thresh; self.lane_tol_ratio = lane_tol_ratio
        self.forward_only_tol = forward_only_tol
        self.duplicate_overlap_thresh = duplicate_overlap_thresh
        self.trackers = []; self.dead_trackers = []
        KalmanBoxTracker.count = 0

    def update(self, dets, frame_idx):
        base_dets = [d for d in dets if not (len(d) > 5 and d[5])]
        relaxed_dets = [d for d in dets if (len(d) > 5 and d[5])]
        filtered_relaxed = []
        for d in relaxed_dets:
            max_overlap = 0.0
            for b in base_dets:
                ov = _candidate_overlap_ratio(b[:4], d[:4])
                if ov > max_overlap: max_overlap = ov
            if max_overlap < self.duplicate_overlap_thresh:
                filtered_relaxed.append(d)
        dets = base_dets + filtered_relaxed
        high_dets = [d for d in dets if d[4] >= self.high_thresh]
        low_dets  = [d for d in dets if self.low_thresh <= d[4] < self.high_thresh]
        for t in self.trackers: t.predict()

        iou1 = _iou_matrix(self.trackers, high_dets)
        if len(self.trackers) > 0 and len(high_dets) > 0:
            fwd_mask1 = _apply_forward_only_mask(self.trackers, high_dets, self.conveyor_vx, self.conveyor_vy, self.forward_only_tol)
            iou1 = np.where(fwd_mask1, iou1, 0.0)
        matched1, unmatched_t1, unmatched_d1 = _hungarian_match(iou1, self.iou_thresh_1)
        for ti, di in matched1:
            self.trackers[ti].update(high_dets[di][:4], high_dets[di][4], frame_idx)

        remain2_t = [self.trackers[i] for i in unmatched_t1]
        remain2_idxs = list(unmatched_t1)
        iou2 = _iou_matrix(remain2_t, low_dets)
        if len(remain2_t) > 0 and len(low_dets) > 0:
            fwd_mask2 = _apply_forward_only_mask(remain2_t, low_dets, self.conveyor_vx, self.conveyor_vy, self.forward_only_tol)
            iou2 = np.where(fwd_mask2, iou2, 0.0)
        matched2, unmatched_t2_local, _ = _hungarian_match(iou2, self.iou_thresh_2)
        matched_t2_set = set(remain2_idxs[li] for li, _ in matched2)
        for li, di in matched2:
            remain2_t[li].update(low_dets[di][:4], low_dets[di][4], frame_idx)

        still_unmatched_t = [self.trackers[i] for i in unmatched_t1 if i not in matched_t2_set]
        still_unmatched_d = [high_dets[i] for i in unmatched_d1]

        if still_unmatched_t and still_unmatched_d:
            lane_mat = _lane_distance_matrix(still_unmatched_t, still_unmatched_d, self.lane_tol_ratio)
            fwd_mask3 = _apply_forward_only_mask(still_unmatched_t, still_unmatched_d, self.conveyor_vx, self.conveyor_vy, self.forward_only_tol)
            lane_mat = np.where(fwd_mask3, lane_mat, np.inf)
            matched3_local = []; used_t, used_d = set(), set()
            flat_indices = np.argsort(lane_mat.ravel())
            for idx in flat_indices:
                ti_l = idx // len(still_unmatched_d); di_l = idx % len(still_unmatched_d)
                if lane_mat[ti_l, di_l] == np.inf: break
                if lane_mat[ti_l, di_l] > self.pos_dist_thresh: break
                if ti_l in used_t or di_l in used_d: continue
                matched3_local.append((ti_l, di_l)); used_t.add(ti_l); used_d.add(di_l)
            for ti_l, di_l in matched3_local:
                d = still_unmatched_d[di_l]
                still_unmatched_t[ti_l].update(d[:4], d[4], frame_idx)
            matched3_dets = {id(still_unmatched_d[di_l]) for _, di_l in matched3_local}
            unmatched_d1_final = [di for di in unmatched_d1 if id(high_dets[di]) not in matched3_dets]
        else:
            unmatched_d1_final = list(unmatched_d1)

        for di in unmatched_d1_final:
            d = high_dets[di]
            new_t = KalmanBoxTracker(d[:4], d[4], init_vx=self.conveyor_vx, init_vy=self.conveyor_vy)
            new_t.history.append((frame_idx, tuple(map(int, d[:4])), d[4]))
            self.trackers.append(new_t)

        alive, dead = [], []
        for t in self.trackers:
            (alive if t.miss <= self.max_miss else dead).append(t)
        self.trackers = alive; self.dead_trackers.extend(dead)

        results = []
        for t in self.trackers:
            if t.miss > 0: continue
            bbox = t.get_bbox(); x1, y1, x2, y2 = map(int, bbox)
            results.append((t.id, x1, y1, x2, y2, t.conf))
        return results

    def get_all_tracks(self):
        all_t = self.trackers + self.dead_trackers
        return {t.id: t.history for t in all_t if t.history}


def merge_tracks(all_tracks, iou_thresh=MERGE_IOU_THRESH, max_frame_gap=MERGE_MAX_FRAME_GAP,
                  conveyor_vx=0.0, conveyor_vy=0.0, partial_area_ratio=PARTIAL_AREA_RATIO,
                  edge_tolerance_ratio=EDGE_TOLERANCE_RATIO,
                  cross_axis_tolerance_ratio=CROSS_AXIS_TOLERANCE_RATIO,
                  min_track_frames_for_merge=MIN_TRACK_FRAMES_FOR_MERGE):
    if len(all_tracks) <= 1:
        return all_tracks

    track_profiles = build_spatial_order(all_tracks, conveyor_vx, conveyor_vy)

    def bbox_area(b): return (b[2]-b[0]) * (b[3]-b[1])
    def bbox_iou(a, b):
        ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
        ix1=max(ax1,bx1); iy1=max(ay1,by1); ix2=min(ax2,bx2); iy2=min(ay2,by2)
        inter = max(0,ix2-ix1)*max(0,iy2-iy1)
        if inter == 0: return 0.0
        ua = (ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
        return inter/ua
    def center_dist(a, b):
        ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
        return np.sqrt(((ax1+ax2)/2-(bx1+bx2)/2)**2 + ((ay1+ay2)/2-(by1+by2)/2)**2)
    def bbox_diag(b):
        x1,y1,x2,y2 = b
        return np.sqrt((x2-x1)**2+(y2-y1)**2)

    track_info = {}
    for tid, hist in all_tracks.items():
        if not hist: continue
        frames_ = [h[0] for h in hist]; bboxes = [h[1] for h in hist]
        areas = [bbox_area(b) for b in bboxes]
        track_info[tid] = {
            'first_frame': min(frames_), 'last_frame': max(frames_),
            'first_bbox': bboxes[0], 'last_bbox': bboxes[-1],
            'max_area': max(areas), 'frames': set(frames_), 'n_frames': len(hist),
        }

    parent = {tid: tid for tid in track_info}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(x, y): parent[find(x)] = find(y)

    speed = np.sqrt(conveyor_vx**2 + conveyor_vy**2)
    is_horizontal = speed < 0.5 or abs(conveyor_vx) >= abs(conveyor_vy)
    tids = list(track_info.keys())

    for i in range(len(tids)):
        for j in range(i+1, len(tids)):
            ti, tj = tids[i], tids[j]
            info_i, info_j = track_info[ti], track_info[tj]
            if info_i['n_frames'] < min_track_frames_for_merge or info_j['n_frames'] < min_track_frames_for_merge:
                continue
            range_overlap = (info_i['first_frame'] < info_j['last_frame'] and info_j['first_frame'] < info_i['last_frame'])
            if range_overlap: continue
            if info_i['last_frame'] <= info_j['first_frame']:
                earlier, later = info_i, info_j; eid, lid = ti, tj
            elif info_j['last_frame'] <= info_i['first_frame']:
                earlier, later = info_j, info_i; eid, lid = tj, ti
            else:
                continue
            dt = later['first_frame'] - earlier['last_frame']
            if dt > max_frame_gap: continue

            is_partial = later['max_area'] < earlier['max_area'] * partial_area_ratio
            if is_partial:
                ex1,ey1,ex2,ey2 = earlier['last_bbox']
                ex1 += conveyor_vx*dt; ex2 += conveyor_vx*dt
                ey1 += conveyor_vy*dt; ey2 += conveyor_vy*dt
                lx1,ly1,lx2,ly2 = later['first_bbox']
                if is_horizontal:
                    e_w = ex2 - ex1
                    edge_tolerance_px = e_w * edge_tolerance_ratio
                    edge_gap = min(abs(ex1-lx1), abs(ex1-lx2), abs(ex2-lx1), abs(ex2-lx2))
                    cross_overlap = min(ey2,ly2) - max(ey1,ly1)
                    cross_tol = -max(ey2-ey1, ly2-ly1) * cross_axis_tolerance_ratio
                    ok = edge_gap <= edge_tolerance_px and cross_overlap > cross_tol
                else:
                    e_h = ey2 - ey1
                    edge_tolerance_px = e_h * edge_tolerance_ratio
                    edge_gap = min(abs(ey1-ly1), abs(ey1-ly2), abs(ey2-ly1), abs(ey2-ly2))
                    cross_overlap = min(ex2,lx2) - max(ex1,lx1)
                    cross_tol = -max(ex2-ex1, lx2-lx1) * cross_axis_tolerance_ratio
                    ok = edge_gap <= edge_tolerance_px and cross_overlap > cross_tol
                if ok: union(eid, lid)
                continue

            prof_e = track_profiles.get(eid); prof_l = track_profiles.get(lid)
            if prof_e and prof_l:
                if not is_same_lane(prof_e, prof_l): continue
                if not depth_order_consistent(prof_e, prof_l, conveyor_vy): continue

            extrap = (earlier['last_bbox'][0]+conveyor_vx*dt, earlier['last_bbox'][1]+conveyor_vy*dt,
                     earlier['last_bbox'][2]+conveyor_vx*dt, earlier['last_bbox'][3]+conveyor_vy*dt)
            first_bbox = later['first_bbox']
            iou = bbox_iou(extrap, first_bbox); dist = center_dist(extrap, first_bbox)
            diag = bbox_diag(earlier['last_bbox'])
            if iou >= iou_thresh or dist <= diag * 0.4:
                union(eid, lid)

    groups = defaultdict(list)
    for tid in tids: groups[find(tid)].append(tid)

    merged = {}
    for root, members in groups.items():
        combined = []
        for tid in members: combined.extend(all_tracks[tid])
        combined.sort(key=lambda x: x[0])
        rep_id = max(members, key=lambda m: track_info[m]['max_area'])
        merged[rep_id] = combined
    return merged


def refine_rail_from_mask(mask, W, H, center_tol=0.35, ransac_residual=15, ratio_min=0.15, ratio_max=0.85):
    mask_bin = (mask > 0).astype(np.uint8)
    ys_all, xs_all = np.where(mask_bin > 0)
    if len(xs_all) == 0:
        return None, "empty"
    cx = xs_all.mean()
    if abs(cx - W/2) / W > center_tol:
        return None, "center"
    left_pts, right_pts = [], []
    y_min, y_max = ys_all.min(), ys_all.max()
    for y in range(y_min, y_max+1):
        xs = np.where(mask_bin[y] > 0)[0]
        if len(xs) < 3: continue
        left_pts.append([xs.min(), y]); right_pts.append([xs.max(), y])
    if len(left_pts) < 10 or len(right_pts) < 10:
        return None, "insufficient_points"

    def fit_line(pts):
        pts = np.array(pts); X = pts[:,1].reshape(-1,1); y = pts[:,0]
        ransac = RANSACRegressor(residual_threshold=ransac_residual, min_samples=10)
        ransac.fit(X, y)
        return ransac

    left_model = fit_line(left_pts); right_model = fit_line(right_pts)
    near_y = H - 1; far_y = y_min
    x_near_left = float(left_model.predict([[near_y]])[0])
    x_near_right = float(right_model.predict([[near_y]])[0])
    x_far_left = float(left_model.predict([[far_y]])[0])
    x_far_right = float(right_model.predict([[far_y]])[0])
    near_w = abs(x_near_right - x_near_left); far_w = abs(x_far_right - x_far_left)
    if far_w >= near_w:
        return None, "perspective_violation"
    ratio = far_w / (near_w + 1e-6)
    if not (ratio_min < ratio < ratio_max):
        return None, "ratio_out_of_range"
    return {
        'near_left': (x_near_left, near_y), 'near_right': (x_near_right, near_y),
        'far_left': (x_far_left, far_y), 'far_right': (x_far_right, far_y),
        'near_width_px': near_w, 'far_width_px': far_w,
    }, "OK"

# ---- [R-3] · [V-0]에서 TRK.이름 으로 부르도록 묶기 ----
TRK_NAMES = ['MOTION_THRESHOLD', 'MOTION_RATIO_THRESH', 'ROI_PAD', 'BASE_MAX_W_RATIO', 'BASE_MAX_H_RATIO', 'RELAXED_MAX_W_RATIO', 'RELAXED_MAX_H_RATIO', 'BASE_MIN_CONF', 'RELAXED_MIN_CONF', 'DUPLICATE_OVERLAP_THRESH', 'LABEL_INTERVAL_SEC', 'CONF_THRESH', 'IOU_THRESH', 'BT_HIGH_THRESH', 'BT_LOW_THRESH', 'BT_IOU_THRESH_1', 'BT_IOU_THRESH_2', 'BT_MAX_MISS', 'MERGE_IOU_THRESH', 'MERGE_MAX_FRAME_GAP', 'POST_NMS_THRESH', 'LANE_TOLERANCE_RATIO', 'DEPTH_DIST_RATIO', 'FORWARD_ONLY_TOLERANCE_RATIO', 'PARTIAL_AREA_RATIO', 'EDGE_TOLERANCE_RATIO', 'CROSS_AXIS_TOLERANCE_RATIO', 'MIN_TRACK_FRAMES_FOR_MERGE', 'MIN_TRACK_FRAMES_FOR_COUNT', 'GLOBAL_MOTION_THRESH', 'GLOBAL_RATIO_THRESH', 'RAIL_N_SAMPLE', 'RAIL_MIN_OK', 'post_nms', 'extract_frames', 'get_roi_for_test_video', 'estimate_conveyor_speed', 'compute_dynamic_params', 'build_spatial_order', 'is_same_lane', 'depth_order_consistent', 'KalmanBoxTracker', '_iou_matrix', '_candidate_overlap_ratio', '_hungarian_match', '_lane_distance_matrix', '_apply_forward_only_mask', 'ByteTracker', 'merge_tracks', 'refine_rail_from_mask']
TRK = types.SimpleNamespace(**{n: globals()[n] for n in TRK_NAMES})
print(f'추적 코드 준비: 상수 · 함수 {len(TRK_NAMES)}개 · 0.1초 간격 · conf {TRK.CONF_THRESH} · ByteTrack high {TRK.BT_HIGH_THRESH} / low {TRK.BT_LOW_THRESH}')

추적 코드 준비: 상수 · 함수 50개 · 0.1초 간격 · conf 0.25 · ByteTrack high 0.5 / low 0.25


In [ ]:
# [R-3] 전체 파이프라인 시간 — 영상 → ROI → YOLO 검출 → 추적 → 레일 → 프레임 선택 · crop → DA3 + DINOv3 → head
#  - 추적 · ROI · 레일 RANSAC은 [R-3a]의 내장 코드(main.py와 같음), 검출 · 레일 세그멘테이션은 같은 가중치의 PyTorch(.pt)
#  - 가중치 파일이 없으면 멈추지 않고 측정만 건너뜀 (저장된 결과가 있으면 그 표를 보여 줌)
#      YOLO (상자): checkpoints/yolo26l_v5_augmentation.pt   ← 캐시를 만든 v5
#      레일 seg  : checkpoints/rail_seg_yolov8s_LabelStudio_best.pt
#  - 먼저 재현 확인: 다시 돌린 트랙 · 레일이 캐시와 같은지 (트랙 수 · 일치율 · 레일 폭 차이) → 같으면 "실제 파이프라인 시간"
#  - YOLO 두 방식: main.py처럼 프레임 한 장씩(batch 1) / 프레임을 YOLO_BS장씩 묶어서 (검출 결과는 같고 시간만 다름)
#  - 실시간 배율 = 영상 하나 전체 처리 시간 ÷ 영상 길이. main.py 추적은 영상 전체를 본 뒤 결과가 나오는 오프라인 방식
#    (ROI · 컨베이어 속도 · 트랙 병합을 영상 전체로 계산) → "처리량 기준 실시간"이며, 상자별 스트리밍 지연은 아님
BOX_PT = f'{CKPT_DIR}/yolo26l_v5_augmentation.pt'
RAIL_PT = f'{CKPT_DIR}/rail_seg_yolov8s_LabelStudio_best.pt'
YOLO_HALF = False                                                    # main.py(ONNX)는 fp32 → 같은 조건 (True면 fp16, 더 빠름)
YOLO_BS = 16
YOLO_KW = {'half': True} if YOLO_HALF else {}                       # fp32면 half 인자를 아예 넘기지 않음 (ultralytics 'half' 지원 종료 경고 방지)
PIPE_READY = os.path.exists(BOX_PT) and os.path.exists(RAIL_PT)      # 가중치가 없으면 측정 · 실시간 처리만 건너뜀 (멈추지 않음)
if PIPE_READY:
    try:
        import ultralytics  # noqa
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
else:
    print(f'⚠️ YOLO 가중치 없음 ({BOX_PT} · {RAIL_PT}) → 전체 파이프라인 측정 · 실시간 처리 시각화는 건너뜀')

if RUN_RUNTIME and PIPE_READY:
    BOX_YOLO, RAIL_YOLO = YOLO(BOX_PT), YOLO(RAIL_PT)                 # 저장 결과를 쓸 때는 [V-0] 실시간 처리에서 필요할 때 로드


def _keep(x1, y1, x2, y2, conf, ch, cw):                               # main.py postprocess와 같은 규칙 (크기 비율 · conf)
    bx1, by1, bx2, by2 = map(int, [x1, y1, x2, y2]); w, h = bx2 - bx1, by2 - by1
    if w <= cw * TRK.BASE_MAX_W_RATIO and h <= ch * TRK.BASE_MAX_H_RATIO:
        return (bx1, by1, bx2, by2, float(conf), False) if conf >= TRK.BASE_MIN_CONF else None
    if w <= cw * TRK.RELAXED_MAX_W_RATIO and h <= ch * TRK.RELAXED_MAX_H_RATIO and conf >= TRK.RELAXED_MIN_CONF:
        return (bx1, by1, bx2, by2, float(conf), True)
    return None


def detect(crops, bs):
    out = []
    for i in range(0, len(crops), bs):
        res = BOX_YOLO.predict(crops[i:i + bs], imgsz=640, conf=TRK.CONF_THRESH, **YOLO_KW, verbose=False, device=0)
        for r, c in zip(res, crops[i:i + bs]):
            ch, cw = c.shape[:2]; b = r.boxes
            kept = [k for k in (_keep(*xy, cf, ch, cw) for xy, cf in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy())) if k]
            out.append(TRK.post_nms(kept))
    return out


def track(all_dets):                                                    # main.py run_two_pass_tracking의 추적 부분
    TRK.KalmanBoxTracker.count = 0; t1 = TRK.ByteTracker()
    for fi, d in enumerate(all_dets): t1.update(d, fi)
    raw1 = t1.get_all_tracks(); vx, vy = TRK.estimate_conveyor_speed(raw1)
    bbs = [h[1] for hist in raw1.values() for h in hist]
    avg_w = float(np.mean([b[2] - b[0] for b in bbs])) if bbs else 150.0
    max_miss, merge_gap = TRK.compute_dynamic_params(vx, vy, TRK.LABEL_INTERVAL_SEC, avg_w)
    TRK.KalmanBoxTracker.count = 0; t2 = TRK.ByteTracker(max_miss=max_miss, conveyor_vx=vx, conveyor_vy=vy)
    for fi, d in enumerate(all_dets): t2.update(d, fi)
    merged = TRK.merge_tracks(t2.get_all_tracks(), max_frame_gap=merge_gap, conveyor_vx=vx, conveyor_vy=vy)
    return {t: h for t, h in merged.items() if len(h) >= TRK.MIN_TRACK_FRAMES_FOR_COUNT}, vx, vy


def rail_quad(frames):                                                  # main.py estimate_rail_quad (세그멘테이션만 PyTorch)
    H, W = frames[0][0].shape[:2]; n = len(frames)
    idxs = np.unique(np.linspace(0, n - 1, min(TRK.RAIL_N_SAMPLE, n)).astype(int))
    res = RAIL_YOLO.predict([frames[i][0] for i in idxs], imgsz=640, conf=0.25, iou=0.45, retina_masks=True,
                            **YOLO_KW, verbose=False, device=0)
    cols = {k: [] for k in ['nl', 'nr', 'fl', 'fr', 'fy']}
    for r in res:
        if r.masks is None or not len(r.boxes): continue
        m = (r.masks.data[int(r.boxes.conf.argmax())].cpu().numpy() > 0.5).astype(np.uint8)
        q, _ = TRK.refine_rail_from_mask(m, W, H)
        if q is None: continue
        cols['nl'].append(q['near_left'][0]); cols['nr'].append(q['near_right'][0])
        cols['fl'].append(q['far_left'][0]); cols['fr'].append(q['far_right'][0]); cols['fy'].append(q['far_left'][1])
    if len(cols['nl']) < TRK.RAIL_MIN_OK: return None
    q = {k: float(np.median(v)) for k, v in cols.items()}
    q.update(near_y=float(H - 1), W=W, H=H, near_w=q['nr'] - q['nl'], far_w=q['fr'] - q['fl'], far_y=q['fy'], n_ok=len(cols['nl']))
    return q


def to_tracks(merged, roi):                                             # 캐시와 같은 형식 (전체 프레임 좌표)
    rx1, ry1 = int(roi[0]), int(roi[1]); out = []
    for tid, hist in merged.items():
        hist = sorted(hist, key=lambda h: h[0])
        out.append({'tid': int(tid), 'fi': np.array([h[0] for h in hist]),
                    'bbox': np.array([[b[0] + rx1, b[1] + ry1, b[2] + rx1, b[3] + ry1] for _, b, _ in hist], float)})
    return out


def compare_cache(tracks, rec):                                         # 캐시 트랙마다 같은 프레임에서 IoU 평균이 0.5 이상인 새 트랙이 있나
    def iou(a, b):
        iw = max(0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0, min(a[3], b[3]) - max(a[1], b[1])); i = iw * ih
        return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)
    ok = 0
    for c in rec['tracks']:
        cm = dict(zip(map(int, c['fi']), c['bbox']))
        best = 0.0
        for t in tracks:
            common = [(cm[f], b) for f, b in zip(map(int, t['fi']), t['bbox']) if f in cm]
            if len(common) >= 0.5 * len(cm):
                best = max(best, np.mean([iou(a, b) for a, b in common]))
        ok += best >= 0.5
    return ok / max(len(rec['tracks']), 1)


def select_and_crop(tracks, frames):                                    # [P-1]과 같은 프레임 선택 · crop (프레임은 메모리에서)
    at = {}
    for t in tracks:
        for fi, b in zip(t['fi'], t['bbox']): at.setdefault(int(fi), []).append((t['tid'], b))
    crops = []
    for t in tracks:
        occ = np.array([occlusion(b, [o for t2, o in at[int(fi)] if t2 != t['tid']]) for fi, b in zip(t['fi'], t['bbox'])])
        for j in select_frames(t['bbox'], occ):
            fr = frames[int(t['fi'][j])][0]; x1, y1, x2, y2 = t['bbox'][j]; H_, W_ = fr.shape[:2]
            px, py = (x2 - x1) * PAD, (y2 - y1) * PAD
            c = fr[int(max(0, y1 - py)):int(min(H_, y2 + py)), int(max(0, x1 - px)):int(min(W_, x2 + px))]
            crops.append(letterbox(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)))
    return np.stack(crops) if crops else np.zeros((0, IMG, IMG, 3), np.uint8)


def run_pipeline(vid, yolo_bs):
    T = {}; _sync(); t0 = time.perf_counter()
    path = video_path('test', vid)
    def lap(k):
        nonlocal t0
        _sync(); t = time.perf_counter(); T[k] = t - t0; t0 = t
    roi = TRK.get_roi_for_test_video(path); lap('ROI')
    frames, fps = TRK.extract_frames(path, TRK.LABEL_INTERVAL_SEC); lap('디코딩')
    rx1, ry1, rx2, ry2 = map(int, roi) if roi is not None else (0, 0, frames[0][0].shape[1], frames[0][0].shape[0])
    dets = detect([f[ry1:ry2, rx1:rx2] for f, _ in frames], yolo_bs); lap('YOLO 검출')
    merged, vx, vy = track(dets); lap('추적')
    quad = rail_quad(frames); lap('레일')
    tracks = to_tracks(merged, (rx1, ry1)); crops = select_and_crop(tracks, frames); lap('프레임 선택 · crop')
    if len(crops): run_heads(encode(crops, ENC_F, BEST_BS, tta=True))
    lap('DA3 + DINOv3 + head')
    return T, tracks, quad, vx, vy, len(frames), len(crops)


# ── 2. 실행 (첫 영상으로 warmup 후 R_VIDS 측정) ────────────────────────────────────────
if RUN_RUNTIME and PIPE_READY:
    run_pipeline(R_VIDS[0], YOLO_BS)
    rows = []
    for vid in R_VIDS:
        dur, _ = video_len_s('test', vid); rec = RECS['test'][vid]
        for mode, bs in (('YOLO 한 장씩 (main.py와 같음)', 1), (f'YOLO {YOLO_BS}장씩 배치', YOLO_BS)):
            T, tracks, quad, vx, vy, n_fr, n_crop = run_pipeline(vid, bs)
            tot = sum(T.values()); n_trk = max(len(tracks), 1)
            row = {'vid': vid, 'YOLO 방식': mode, '영상 길이 s': dur, '샘플 프레임': n_fr, '트랙 (캐시)': len(rec['tracks']),
                   '트랙 (다시 돌림)': len(tracks), '캐시 트랙 일치율 %': compare_cache(tracks, rec) * 100,
                   '레일 폭 차이 %': (abs(quad['near_w'] - rec['quad']['near_w']) / rec['quad']['near_w'] * 100) if quad and rec['quad'] else np.nan,
                   **{f'{k} s': v for k, v in T.items()}, '전체 s': tot, '박스당 ms': tot / n_trk * 1000, '실시간 배율': tot / dur}
            rows.append(row)
        print(f'  {vid}: {dur:.1f}s · 트랙 캐시 {len(rec["tracks"])} / 다시 {len(tracks)} · 일치율 {rows[-1]["캐시 트랙 일치율 %"]:.0f}%')
    PIPE = pd.DataFrame(rows)
else:
    PIPE = pd.DataFrame(RJ['pipeline']) if RJ.get('pipeline') else None
if PIPE is not None and len(PIPE):
    hr(f'▶ [R-3] 전체 파이프라인 — 재현 확인 ({ENV["GPU"]})')
    display(PIPE[PIPE['YOLO 방식'] == PIPE['YOLO 방식'].iloc[0]].set_index('vid')[
        ['영상 길이 s', '샘플 프레임', '트랙 (캐시)', '트랙 (다시 돌림)', '캐시 트랙 일치율 %', '레일 폭 차이 %']].round(1))
    hr('▶ [R-3] 전체 파이프라인 — 단계별 시간 (영상 평균, 초) · 박스당 · 실시간 배율')
    STAGES = [c for c in PIPE.columns if c.endswith(' s') and c not in ('영상 길이 s', '전체 s')]
    PG = PIPE.groupby('YOLO 방식', sort=False).agg({**{c: 'mean' for c in STAGES + ['전체 s', '영상 길이 s', '박스당 ms']},
                                                    '실시간 배율': ['mean', 'max']})
    PG.columns = [f'{a} ({b})' if a == '실시간 배율' else a for a, b in PG.columns]
    display(PG.round(3))
    print('※ 일치율이 높고 레일 폭 차이가 작으면 캐시를 만든 파이프라인과 같은 동작 · PyTorch(.pt) 실행이라 ONNX와 검출 결과가 아주 조금 다를 수 있음')
else:
    STAGES = []; print('⏭ 전체 파이프라인 결과 없음 (YOLO 가중치가 없거나 측정하지 않음) → [R-4]는 치수 추정 부분만 요약')


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
  test_025: 5.3s · 트랙 캐시 2 / 다시 2 · 일치율 100%
  test_012: 5.3s · 트랙 캐시 5 / 다시 5 · 일치율 100%
  test_008: 5.9s · 트랙 캐시 6 / 다시 6 · 일치율 100%
  test_030: 8.9s · 트랙 캐시 8 / 다시 8 · 일치율 100%
  test_041: 10.1s · 트랙 캐시 9 / 다시 10 · 일치율 100%
  test_022: 9.5s · 트랙 캐시 10 / 다시 10 · 일치율 100%
  test_005: 14.3s · 트랙 캐시 11 / 다시 11 · 일치율 100%
  test_026: 12.5s · 트랙 캐시 12 / 다시 12 · 일치율 100%
  test_004: 10.7s · 트랙 캐시 15 / 다시 16 · 일치율 93%
  test_038: 15.5s · 트랙 캐시 20 / 다시 22 · 일치율 100%

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ [R-3] 전체 파이프라인 — 재현 확인 (NVIDIA A100-SXM4-40GB)


,영상 길이 s,샘플 프레임,트랙 (캐시),트랙 (다시 돌림),캐시 트랙 일치율 %,레일 폭 차이 %
vid,,,,,,
test_025,5.3,53,2,2,100.0,0.1
test_012,5.3,53,5,5,100.0,0.2
test_008,5.9,59,6,6,100.0,0.0
test_030,8.9,89,8,8,100.0,0.1
test_041,10.1,101,9,10,100.0,0.0
test_022,9.5,95,10,10,100.0,0.0
test_005,14.3,143,11,11,100.0,0.2
test_026,12.5,125,12,12,100.0,0.0
test_004,10.7,107,15,16,93.3,0.0



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ [R-3] 전체 파이프라인 — 단계별 시간 (영상 평균, 초) · 박스당 · 실시간 배율


,ROI s,디코딩 s,YOLO 검출 s,추적 s,레일 s,프레임 선택 · crop s,DA3 + DINOv3 + head s,전체 s,영상 길이 s,박스당 ms,실시간 배율 (mean),실시간 배율 (max)
YOLO 방식,,,,,,,,,,,,
YOLO 한 장씩 (main.py와 같음),1.834,0.265,1.999,0.160,0.630,0.045,1.736,6.669,9.767,808.119,0.704,0.842
YOLO 16장씩 배치,1.827,0.253,0.587,0.158,0.623,0.045,1.710,5.202,9.767,624.197,0.552,0.718


※ 일치율이 높고 레일 폭 차이가 작으면 캐시를 만든 파이프라인과 같은 동작 · PyTorch(.pt) 실행이라 ONNX와 검출 결과가 아주 조금 다를 수 있음


In [ ]:
# [R-4] 런타임 요약 — 배치 처리 효과 · 박스당 시간 · 실시간 여부 (보고서 표)
S = RT.mean(numeric_only=True); W = RT.max(numeric_only=True)
SUM = pd.DataFrame({
    '박스당 시간 ms (평균)': [S['기존 ms/박스'], S['배치 ms/박스'], S['스트리밍 지연 ms/박스 (평균)']],
    '실시간 배율 (평균)': [S['실시간 배율 기존'], S['실시간 배율 배치'], S['실시간 배율 스트리밍']],
    '실시간 배율 (가장 느린 영상)': [W['실시간 배율 기존'], W['실시간 배율 배치'], W['실시간 배율 스트리밍']],
}, index=['치수 추정만 · 기존 (seek 읽기 · 한 장씩)', '치수 추정만 · 배치 (순차 읽기 · 영상 단위 배치)',
          '치수 추정만 · 스트리밍 (트랙 단위 배치, 읽기 없음)'])
_pipe = PIPE if (PIPE is not None and len(PIPE)) else pd.DataFrame(columns=['YOLO 방식'])
for mode, g in _pipe.groupby('YOLO 방식', sort=False):                    # [R-3] 전체 파이프라인 (있을 때만)
    SUM.loc[f'전체 파이프라인 · {mode}'] = [g['박스당 ms'].mean(), g['실시간 배율'].mean(), g['실시간 배율'].max()]
SUM['실시간'] = np.where(SUM['실시간 배율 (가장 느린 영상)'] < 1, '✅', '✗')
hr(f'▶ [R-4] 런타임 요약 — {ENV["GPU"]} · fp16 (치수) · 448 · 트랙당 최대 8장 · TTA · test 영상 {len(RT)}개')
display(SUM.round(3))
fp = _pipe.groupby('YOLO 방식', sort=False)
print(f'- 치수 추정 배치 효과: 박스당 {S["기존 ms/박스"]:.0f} → {S["배치 ms/박스"]:.0f} ms ({S["기존 ms/박스"] / S["배치 ms/박스"]:.1f}배)'
      f'  · 읽기 {S["읽기 seek ms/박스"]:.0f} → {S["읽기 순차 ms/박스"]:.0f} ms · 모델 {S["모델 한 장씩 ms/박스"]:.0f} → {S["모델 배치 ms/박스"]:.0f} ms')
print(f'- crop당: batch 1 {TB.loc[1, "DA3 + DINOv3 (최종)"]:.1f} ms → batch {BEST_BS} {TB.loc[BEST_BS, "DA3 + DINOv3 (최종)"]:.1f} ms (TTA 없음, 배치는 GPU가 포화되는 크기까지)')
print(f'- 박스당 지연 (치수 추정 스트리밍): 평균 {S["스트리밍 지연 ms/박스 (평균)"]:.0f} ms · 최대 {W["스트리밍 지연 ms/박스 (최대)"]:.0f} ms'
      f'  vs 평균 상자 간격 {S["평균 상자 간격 s"]:.2f} s')
for mode, g in fp:
    st = g[STAGES].mean(); top = st.idxmax()
    print(f'- 전체 파이프라인 ({mode}): 영상당 {g["전체 s"].mean():.1f} s / 길이 {g["영상 길이 s"].mean():.1f} s'
          f' → 실시간 배율 평균 {g["실시간 배율"].mean():.2f} · 최대 {g["실시간 배율"].max():.2f} · 가장 오래 걸린 단계: {top[:-2]} ({st.max():.1f} s)')
print('- 전체 파이프라인은 main.py 추적 방식(영상 전체를 본 뒤 결과)이라 처리량 기준 실시간이다 (상자별 스트리밍 지연이 아님)')
if RUN_RUNTIME:                                                         # 새로 측정했을 때만 저장 (덮어씀)
    with open(RUNTIME_JSON, 'w') as f_:
        json.dump({'env': {k: str(v) for k, v in ENV.items()}, 'best_batch': BEST_BS, 'batch_table': TB.reset_index().to_dict('records'),
                   'per_video': RT.reset_index().to_dict('records'), 'pipeline': _pipe.to_dict('records'),
                   'summary': SUM.reset_index().to_dict('records')}, f_, ensure_ascii=False, indent=1, default=float)
    print(f'💾 {RUNTIME_JSON}')
else:
    print(f'⏭ 저장된 결과를 다시 보여 줌 ({RUNTIME_JSON}) — 새로 재려면 [R-0]에서 RUN_RUNTIME = True')
for _n in ('V3', 'DA3_ENC', 'HEADS', 'BOX_YOLO', 'RAIL_YOLO'):
    globals().pop(_n, None)
gc.collect(); torch.cuda.empty_cache()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ [R-4] 런타임 요약 — NVIDIA A100-SXM4-40GB · fp16 (치수) · 448 · 트랙당 최대 8장 · TTA · test 영상 10개


,박스당 시간 ms (평균),실시간 배율 (평균),실시간 배율 (가장 느린 영상),실시간
치수 추정만 · 기존 (seek 읽기 · 한 장씩),1086.495,1.038,1.413,✗
치수 추정만 · 배치 (순차 읽기 · 영상 단위 배치),192.329,0.183,0.243,✅
"치수 추정만 · 스트리밍 (트랙 단위 배치, 읽기 없음)",192.442,0.185,0.253,✅
전체 파이프라인 · YOLO 한 장씩 (main.py와 같음),808.119,0.704,0.842,✅
전체 파이프라인 · YOLO 16장씩 배치,624.197,0.552,0.718,✅


- 치수 추정 배치 효과: 박스당 1086 → 192 ms (5.6배)  · 읽기 270 → 19 ms · 모델 817 → 173 ms
- crop당: batch 1 49.0 ms → batch 64 11.0 ms (TTA 없음, 배치는 GPU가 포화되는 크기까지)
- 박스당 지연 (치수 추정 스트리밍): 평균 192 ms · 최대 196 ms  vs 평균 상자 간격 1.17 s
- 전체 파이프라인 (YOLO 한 장씩 (main.py와 같음)): 영상당 6.7 s / 길이 9.8 s → 실시간 배율 평균 0.70 · 최대 0.84 · 가장 오래 걸린 단계: YOLO 검출 (2.0 s)
- 전체 파이프라인 (YOLO 16장씩 배치): 영상당 5.2 s / 길이 9.8 s → 실시간 배율 평균 0.55 · 최대 0.72 · 가장 오래 걸린 단계: ROI (1.8 s)
- 전체 파이프라인은 main.py 추적 방식(영상 전체를 본 뒤 결과)이라 처리량 기준 실시간이다 (상자별 스트리밍 지연이 아님)
💾 /content/drive/MyDrive/4학년 1학기 과목별 정리/소프트웨어융합캡스톤디자인/BoxSizing/outputs/step2/runtime_summary.json


In [ ]:
# [R-3c] 고정 카메라 — ROI · 레일을 설치할 때 한 번만 구했다고 보고, 영상마다 나머지 단계만 측정
#  - 현장 CCTV는 고정이라 ROI(컨베이어 영역)와 레일 quad가 바뀌지 않는다 → 설치할 때 한 번 계산해 두고 재사용
#  - 여기서는 영상마다 카메라가 다르므로, 영상마다 [설치 1회] ROI · 레일을 먼저 구해 두고 (시간은 따로 기록)
#    [영상 처리] 디코딩 → YOLO 검출 → 추적 → 프레임 선택 · crop → DA3 + DINOv3 + head 만 시간을 잰다
#  - [R-3]과 같은 영상 · 같은 함수 · 같은 YOLO 두 방식 · 첫 영상으로 warmup. ROI가 같으므로 검출 · 트랙도 [R-3]과 같아야 한다
#  - 시간만 재는 셀이다 (정확도 채점은 [T-1]에서 끝남)
#  - 결과는 runtime_summary.json에 'pipeline_fixed'로 추가 저장 (기존 결과는 그대로)

# ── 0. 준비 — [R-4]가 메모리를 비우며 지운 모델 5개(YOLO 2 · DINOv3 · DA3 인코더 · head)를 다시 불러옴 ──
assert PIPE_READY, f'YOLO 가중치(.pt) 없음 → {BOX_PT} · {RAIL_PT}'
if 'BOX_YOLO' not in globals() or 'RAIL_YOLO' not in globals():
    from ultralytics import YOLO
    BOX_YOLO, RAIL_YOLO = YOLO(BOX_PT), YOLO(RAIL_PT)
if 'V3' not in globals():
    load_v3()                                                           # DINOv3-L ([R-1]의 함수)
if 'DA3_ENC' not in globals():
    DA3_ENC = da3_encoder()                                             # DA3 인코더 ([T-0b]의 함수)
if not globals().get('HEADS'):
    HEADS = []                                                          # head 10개 ([R-2]와 같음)
    for cfg in (STEP1, STEP2):
        for k in range(N_FOLD):
            ck = torch.load(f"{CKPT2}/{cfg['name']}_fold{k}.pt", map_location='cpu', weights_only=False)
            h = TokHead(np.zeros(len(cfg['tgt'])), len(cfg['side']), 49, 2048).to(DEV).eval(); h.load_state_dict(ck['state'])
            HEADS.append((h, len(cfg['side'])))
_miss = [n for n in ['R_VIDS', 'encode', 'ENC_F', 'BEST_BS', 'run_heads', 'PIPE', 'detect', 'track', 'rail_quad',
                     'to_tracks', 'select_and_crop', 'compare_cache', 'video_len_s', '_sync'] if n not in globals()]
assert not _miss, f'없는 변수 {_miss} → [R-0](RUN_RUNTIME = True) · [R-1] · [R-2] · [R-3]을 먼저 실행'
print(f'모델 준비 완료: YOLO 2 · DINOv3 · DA3 인코더 · head {len(HEADS)}개')


def setup_camera(vid):
    """설치 1회: ROI + 레일 quad (레일은 영상 프레임으로 추정)"""
    T = {}; _sync(); t0 = time.perf_counter(); path = video_path('test', vid)
    roi = TRK.get_roi_for_test_video(path); _sync(); T['ROI s'] = time.perf_counter() - t0; t0 = time.perf_counter()
    frames, _ = TRK.extract_frames(path, TRK.LABEL_INTERVAL_SEC)
    quad = rail_quad(frames); _sync(); T['레일 (디코딩 포함) s'] = time.perf_counter() - t0
    return roi, quad, T


def run_fixed(vid, yolo_bs, roi):
    """영상 처리: ROI · 레일 계산 없이 나머지 단계 (단계 이름은 [R-3]과 같음)"""
    T = {}; _sync(); t0 = time.perf_counter(); path = video_path('test', vid)
    def lap(k):
        nonlocal t0
        _sync(); t = time.perf_counter(); T[k] = t - t0; t0 = t
    frames, fps = TRK.extract_frames(path, TRK.LABEL_INTERVAL_SEC); lap('디코딩')
    rx1, ry1, rx2, ry2 = map(int, roi) if roi is not None else (0, 0, frames[0][0].shape[1], frames[0][0].shape[0])
    dets = detect([f[ry1:ry2, rx1:rx2] for f, _ in frames], yolo_bs); lap('YOLO 검출')
    merged, vx, vy = track(dets); lap('추적')
    tracks = to_tracks(merged, (rx1, ry1)); crops = select_and_crop(tracks, frames); lap('프레임 선택 · crop')
    if len(crops): run_heads(encode(crops, ENC_F, BEST_BS, tta=True))
    lap('DA3 + DINOv3 + head')
    return T, tracks, len(crops)


# ── 1. 설치 1회 (영상마다 = 카메라마다) ───────────────────────────────────────
SETUP = {}
setup_camera(R_VIDS[0])                                                    # warmup
for vid in R_VIDS:
    roi, quad, T = setup_camera(vid)
    SETUP[vid] = {'roi': roi, 'quad': quad, **T}

# ── 2. 영상 처리 (첫 영상으로 warmup 후 측정) ─────────────────────────────────
run_fixed(R_VIDS[0], YOLO_BS, SETUP[R_VIDS[0]]['roi'])
rows = []
for vid in R_VIDS:
    dur, _ = video_len_s('test', vid); rec = RECS['test'][vid]
    for mode, bs in (('YOLO 한 장씩 (main.py와 같음)', 1), (f'YOLO {YOLO_BS}장씩 배치', YOLO_BS)):
        T, tracks, n_crop = run_fixed(vid, bs, SETUP[vid]['roi'])
        tot = sum(T.values()); n_trk = max(len(tracks), 1)
        rows.append({'vid': vid, 'YOLO 방식': mode, '영상 길이 s': dur, '트랙 (캐시)': len(rec['tracks']), '트랙 (다시 돌림)': len(tracks),
                     '캐시 트랙 일치율 %': compare_cache(tracks, rec) * 100, **{f'{k} s': v for k, v in T.items()},
                     '전체 s': tot, '박스당 ms': tot / n_trk * 1000, '실시간 배율': tot / dur,
                     '설치 1회 s': SETUP[vid]['ROI s'] + SETUP[vid]['레일 (디코딩 포함) s']})
    print(f'  {vid}: {dur:.1f}s · 트랙 {len(tracks)} (캐시 {len(rec["tracks"])}) · 일치율 {rows[-1]["캐시 트랙 일치율 %"]:.0f}% · '
          f'영상 처리 {tot:.1f}s (배치) · 설치 1회 {rows[-1]["설치 1회 s"]:.1f}s')
PIPE_FIXED = pd.DataFrame(rows)

# ── 3. 결과 ───────────────────────────────────────────────────────────────
hr(f'▶ [R-3c] 고정 카메라 — 영상 처리 단계별 시간 (영상 평균, 초) · 박스당 · 실시간 배율 ({ENV["GPU"]})')
ST_F = [c for c in PIPE_FIXED.columns if c.endswith(' s') and c not in ('영상 길이 s', '전체 s', '설치 1회 s')]
PF = PIPE_FIXED.groupby('YOLO 방식', sort=False).agg({**{c: 'mean' for c in ST_F + ['전체 s', '영상 길이 s', '박스당 ms', '설치 1회 s']},
                                                      '실시간 배율': ['mean', 'max']})
PF.columns = [f'{a} ({b})' if a == '실시간 배율' else a for a, b in PF.columns]
display(PF.round(3))
S_SET = pd.DataFrame(SETUP).T[['ROI s', '레일 (디코딩 포함) s']].astype(float)
print(f'설치 1회 (카메라마다 한 번): ROI 평균 {S_SET["ROI s"].mean():.2f} s · 레일 평균 {S_SET["레일 (디코딩 포함) s"].mean():.2f} s')

hr('▶ [R-3] (영상마다 ROI · 레일 계산) vs [R-3c] (고정 카메라) — 영상 처리 시간 · 실시간 배율')
cmp = []
for mode in PF.index:
    a = PIPE[PIPE['YOLO 방식'] == mode]; b = PIPE_FIXED[PIPE_FIXED['YOLO 방식'] == mode]
    for name, d in (('영상마다 ROI · 레일 계산 [R-3]', a), ('고정 카메라 [R-3c]', b)):
        cmp.append({'YOLO 방식': mode, '조건': name, '영상당 s': d['전체 s'].mean(), '박스당 ms': d['박스당 ms'].mean(),
                    '실시간 배율 (평균)': d['실시간 배율'].mean(), '실시간 배율 (가장 느린 영상)': d['실시간 배율'].max(),
                    '트랙 수 합': int(d['트랙 (다시 돌림)'].sum())})
CMP = pd.DataFrame(cmp).set_index(['YOLO 방식', '조건'])
CMP['실시간'] = np.where(CMP['실시간 배율 (가장 느린 영상)'] < 1, '✅', '✗')
display(CMP.round(3))
same = (PIPE.groupby('YOLO 방식')['트랙 (다시 돌림)'].sum() == PIPE_FIXED.groupby('YOLO 방식')['트랙 (다시 돌림)'].sum()).all()
print('트랙 수가 [R-3]과 같음 (같은 ROI → 같은 검출 · 추적):', '✅' if same else '⚠️ 다름 — 확인 필요')

# ── 4. 저장 (runtime_summary.json에 추가, 기존 항목은 그대로) ─────────────────────
try:
    with open(RUNTIME_JSON) as f_: J = json.load(f_)
except FileNotFoundError:
    J = {}
J['pipeline_fixed'] = PIPE_FIXED.to_dict('records')
J['setup_fixed'] = S_SET.reset_index().rename(columns={'index': 'vid'}).to_dict('records')
J['pipeline_compare'] = CMP.reset_index().to_dict('records')
with open(RUNTIME_JSON, 'w') as f_:
    json.dump(J, f_, ensure_ascii=False, indent=1, default=float)
print(f'💾 {RUNTIME_JSON} (pipeline_fixed · setup_fixed · pipeline_compare 추가)')

# ── 5. 메모리 정리 ([R-4]와 같음) ──────────────────────────────────────────────
for _n in ('V3', 'DA3_ENC', 'HEADS', 'BOX_YOLO', 'RAIL_YOLO'):
    globals().pop(_n, None)
gc.collect(); torch.cuda.empty_cache()

모델 준비 완료: YOLO 2 · DINOv3 · DA3 인코더 · head 10개
  test_025: 5.3s · 트랙 2 (캐시 2) · 일치율 100% · 영상 처리 0.8s (배치) · 설치 1회 2.2s
  test_012: 5.3s · 트랙 5 (캐시 5) · 일치율 100% · 영상 처리 1.4s (배치) · 설치 1회 2.4s
  test_008: 5.9s · 트랙 6 (캐시 6) · 일치율 100% · 영상 처리 1.5s (배치) · 설치 1회 2.4s
  test_030: 8.9s · 트랙 8 (캐시 8) · 일치율 100% · 영상 처리 2.1s (배치) · 설치 1회 2.8s
  test_041: 10.1s · 트랙 10 (캐시 9) · 일치율 100% · 영상 처리 2.6s (배치) · 설치 1회 2.3s
  test_022: 9.5s · 트랙 10 (캐시 10) · 일치율 100% · 영상 처리 2.7s (배치) · 설치 1회 2.5s
  test_005: 14.3s · 트랙 11 (캐시 11) · 일치율 100% · 영상 처리 3.2s (배치) · 설치 1회 3.3s
  test_026: 12.5s · 트랙 12 (캐시 12) · 일치율 100% · 영상 처리 3.1s (배치) · 설치 1회 2.7s
  test_004: 10.7s · 트랙 16 (캐시 15) · 일치율 93% · 영상 처리 3.6s (배치) · 설치 1회 2.7s
  test_038: 15.5s · 트랙 22 (캐시 20) · 일치율 100% · 영상 처리 5.3s (배치) · 설치 1회 3.6s

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ [R-3c] 고정 카메라 — 영상 처리 단계별 시간 (영상 평균, 초) · 박스당 · 실시간 배율 (NVIDIA A100-SXM4-40GB)


,디코딩 s,YOLO 검출 s,추적 s,프레임 선택 · crop s,DA3 + DINOv3 + head s,전체 s,영상 길이 s,박스당 ms,설치 1회 s,실시간 배율 (mean),실시간 배율 (max)
YOLO 방식,,,,,,,,,,,
YOLO 한 장씩 (main.py와 같음),0.244,1.930,0.162,0.046,1.732,4.114,9.767,446.433,2.693,0.412,0.489
YOLO 16장씩 배치,0.246,0.479,0.161,0.046,1.708,2.639,9.767,275.042,2.693,0.262,0.341


설치 1회 (카메라마다 한 번): ROI 평균 1.81 s · 레일 평균 0.88 s

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
▶ [R-3] (영상마다 ROI · 레일 계산) vs [R-3c] (고정 카메라) — 영상 처리 시간 · 실시간 배율


영상당 s   박스당 ms  실시간 배율 (평균)  \
YOLO 방식                 조건                                                    
YOLO 한 장씩 (main.py와 같음) 영상마다 ROI · 레일 계산 [R-3]  6.669  808.119        0.704   
                        고정 카메라 [R-3c]           4.114  446.433        0.412   
YOLO 16장씩 배치            영상마다 ROI · 레일 계산 [R-3]  5.202  624.197        0.552   
                        고정 카메라 [R-3c]           2.639  275.042        0.262   

                                                실시간 배율 (가장 느린 영상)  트랙 수 합 실시간  
YOLO 방식                 조건                                                     
YOLO 한 장씩 (main.py와 같음) 영상마다 ROI · 레일 계산 [R-3]              0.842     102   ✅  
                        고정 카메라 [R-3c]                       0.489     102   ✅  
YOLO 16장씩 배치            영상마다 ROI · 레일 계산 [R-3]              0.718     102   ✅  
                        고정 카메라 [R-3c]                       0.341     102   ✅

트랙 수가 [R-3]과 같음 (같은 ROI → 같은 검출 · 추적): ✅
💾 /content/drive/MyDrive/4학년 1학기 과목별 정리/소프트웨어융합캡스톤디자인/BoxSizing/outputs/step2/runtime_summary.json (pipeline_fixed · setup_fixed · pipeline_compare 추가)


# **10. 시각화 — test 영상 위에 최종 결과 + 상자별 정답 비교**
- **[V-0]** 영상 번호(`VIS_NUM`)와 방식(`VIS_MODE`)을 고른다
  - **실시간 처리:** 영상 파일만 넣고 파이프라인 전체(ROI → YOLO → 추적 → 레일 → 프레임 선택 → DA3 + DINOv3 → head)를 이 자리에서 새로 돌린다. 트래킹 캐시 · 저장된 예측을 쓰지 않는다 ([R-3]을 먼저 실행해야 함). 정답 비교는 새 트랙을 캐시 트랙과 bbox IoU로 맞춘 뒤 수작업 매핑으로 붙인다
  - **저장된 결과:** 트래킹 캐시 + [F-2] 예측(test 채점에 쓴 값)을 그대로 그린다 ([R-3] 없이도 동작)
- **[V-1]** mp4: 위는 영상 위 bbox · 예측 · 정답 · 오차, 아래는 **모든 상자의 w · d · h 예측(주황) vs 정답(초록) 막대 · 오차** (지금 화면에 보이는 상자는 밝게)
- **[V-2]** 상자별 표 + 대표 crop · 가상 3D 상자 (예측 주황 · 정답 초록 점선)

In [ ]:
# [V-0] 시각화할 test 영상 고르기 — 저장된 결과 (트래킹 캐시 + [F-2] 예측표, test 채점에 쓴 바로 그 값)
#  - 영상 번호(VIS_NUM)만 바꿔서 [V-0] → [V-1] → [V-2] 순서로 실행
VIS_NUM = 5  #@param {type:"integer"}
VIS_VID = f'test_{VIS_NUM:03d}'
assert VIS_VID in RECS['test'], f'{VIS_VID} 없음 (0 ~ {len(RECS["test"]) - 1})'
fmt = lambda a: 'x'.join(f'{v:.0f}' for v in a)

VREC = RECS['test'][VIS_VID]
VPRED = PRED['test'][PRED['test'].vid == VIS_VID].set_index('tid')
VFR = pd.concat([FR[('test', p)] for p in ('mapped', 'unmapped') if FR[('test', p)] is not None])
VFR = VFR[VFR.vid == VIS_VID]
print(f'{VIS_VID}: 트랙 {len(VPRED)}개 · 정답과 연결 {int(VPRED.mapped.sum())}개 · focal {VPRED.focal_mm.iloc[0]:.1f} mm')

test_005: 트랙 11개 · 정답과 연결 11개 · focal 12.7 mm


In [ ]:
# [V-1] 영상 위에 결과를 그려 mp4로 저장 (재생과 상자별 표는 [V-2])
#  - 초록 = 정답과 연결된 상자 · 주황 = 정답 없는 트랙 · 굵은 노란 테두리 = 그 트랙의 치수 추론에 쓴 프레임 (가려짐 적은 최대 8장)
#  - 글자는 cv2가 한글을 못 그려서 영어 (pred = 예측, GT = 정답, err = 치수 오차)
#  - 트래킹 캐시는 3프레임마다 저장 → 그 프레임만 모아 약 10 fps 영상으로 만든다
VIS_WIDTH = 960  #@param {type:"integer"}
import imageio.v2 as imageio
VIS_DIR = f'{OUT2_DIR}/vis'; os.makedirs(VIS_DIR, exist_ok=True)
step = int(VREC['step']); FONT = cv2.FONT_HERSHEY_SIMPLEX
USED = {(int(t), int(fi)) for t, fi in VFR[['tid', 'rep_fi']].itertuples(index=False)}
INFO = {}
for tid, r in VPRED.iterrows():
    pred = np.array([r.w, r.d, r.h])
    if r.mapped:
        gt = np.array([r.w_gt, r.d_gt, r.h_gt]); err = np.abs(sort3(pred) - sort3(gt)).mean()
        INFO[int(tid)] = ((60, 200, 60), [f'T{tid} pred {fmt(pred)}', f'GT {fmt(gt)}  err {err:.1f}cm'])
    else:
        INFO[int(tid)] = ((0, 150, 255), [f'T{tid} pred {fmt(pred)}', 'no GT'])
AT = {}
for t in VREC['tracks']:
    if int(t['tid']) in INFO:
        for fi, b in zip(t['fi'], t['bbox']): AT.setdefault(int(fi), []).append((int(t['tid']), b))

cap = cv2.VideoCapture(video_path('test', VIS_VID)); fps_v = cap.get(cv2.CAP_PROP_FPS)
head = f'{VIS_VID} | focal {VPRED.focal_mm.iloc[0]:.1f}mm | tracks {len(INFO)}'
VIS_PATH = f'{VIS_DIR}/{VIS_VID}_result.mp4'; frames_out, i = [], 0
while True:
    ok_, fr = cap.read()
    if not ok_: break
    if i % step == 0:
        fi = i // step
        for tid, b in AT.get(fi, []):
            col, lines = INFO[tid]; x1, y1, x2, y2 = map(int, b); used = (tid, fi) in USED
            cv2.rectangle(fr, (x1, y1), (x2, y2), (0, 230, 255) if used else col, 4 if used else 2)
            for j, txt in enumerate(lines):
                yy = max(18, y1 - 8 - 18 * (len(lines) - 1 - j))
                (tw, th), _ = cv2.getTextSize(txt, FONT, 0.5, 1)
                cv2.rectangle(fr, (x1, yy - th - 4), (x1 + tw + 4, yy + 3), (0, 0, 0), -1)
                cv2.putText(fr, txt, (x1 + 2, yy), FONT, 0.5, col, 1, cv2.LINE_AA)
        cv2.rectangle(fr, (0, 0), (fr.shape[1], 28), (0, 0, 0), -1)
        cv2.putText(fr, f'{head} | frame {i}', (8, 20), FONT, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
        s = VIS_WIDTH / fr.shape[1]
        frames_out.append(cv2.cvtColor(cv2.resize(fr, (VIS_WIDTH, int(fr.shape[0] * s) // 2 * 2)), cv2.COLOR_BGR2RGB))
    i += 1
cap.release()
imageio.mimsave(VIS_PATH, frames_out, fps=fps_v / step, codec='libx264', quality=7, macro_block_size=2)
print(f'💾 {VIS_PATH} ({len(frames_out)} 프레임 · {fps_v / step:.1f} fps)')



💾 /content/drive/MyDrive/4학년 1학기 과목별 정리/소프트웨어융합캡스톤디자인/BoxSizing/outputs/step2/vis/test_005_result.mp4 (143 프레임 · 10.0 fps)


In [ ]:
# [V-2] 영상 재생 + 바로 아래에 상자별 결과 표 · 대표 crop · 가상 3D 상자 (예측 주황 · 정답 초록 점선, 같은 축척)
from IPython.display import Video


def cuboid(ax, w, d, h, color, ls, label):
    V = np.array([[x, y, z] for x in (0, w) for y in (0, d) for z in (0, h)])
    for a in range(8):
        for b in range(a + 1, 8):
            if np.sum(V[a] != V[b]) == 1:
                ax.plot(*zip(V[a], V[b]), color=color, ls=ls, lw=2, label=label if (a, b) == (0, 1) else None)


# ① 영상
display(Video(VIS_PATH, embed=True, width=VIS_WIDTH))

# ② 상자별 표
rows = []
for tid, r in VPRED.iterrows():
    pred = np.array([r.w, r.d, r.h]); gt = np.array([r.w_gt, r.d_gt, r.h_gt]) if r.mapped else None
    rows.append({'트랙': int(tid), '예측 w×d×h (cm)': fmt(pred), '정답 w×d×h (cm)': fmt(gt) if gt is not None else '—',
                 '치수 오차 cm': np.abs(sort3(pred) - sort3(gt)).mean() if gt is not None else np.nan,
                 '치수 오차 %': (np.abs(sort3(pred) - sort3(gt)) / sort3(gt)).mean() * 100 if gt is not None else np.nan,
                 '추론 프레임 수': int(r.n_frames)})
TBL = pd.DataFrame(rows).set_index('트랙')
hr(f'▶ {VIS_VID} — 상자별 결과 (정답 있는 상자 평균 {TBL["치수 오차 cm"].mean():.2f} cm · {TBL["치수 오차 %"].mean():.1f}%)')
display(TBL.round(2))

# ③ 대표 crop (추론에 쓴 프레임 중 가려짐이 가장 적은 것) · 가상 3D 상자
FV = VFR.sort_values(['tid', 'occ']); tids = list(VPRED.index); ncol = 4; nrow = math.ceil(len(tids) * 2 / ncol)
fig = plt.figure(figsize=(4.2 * ncol, 3.8 * nrow)); cap = cv2.VideoCapture(video_path('test', VIS_VID))
for j, tid in enumerate(tids):
    r = VPRED.loc[tid]; fr_row = FV[FV.tid == tid].iloc[0]
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fr_row.rep_fi) * step); ok_, fr = cap.read()
    ax = fig.add_subplot(nrow, ncol, 2 * j + 1)
    if ok_:
        H_, W_ = fr.shape[:2]; px, py = fr_row.bw * 0.4, fr_row.bh * 0.4
        c = fr[int(max(0, fr_row.y1 - py)):int(min(H_, fr_row.y2 + py)), int(max(0, fr_row.x1 - px)):int(min(W_, fr_row.x2 + px))]
        ax.imshow(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
    ax.axis('off'); ax.set_title(f'T{tid} · 가려짐 {fr_row.occ:.2f}', fontsize=10)
    pred = np.array([r.w, r.d, r.h]); gt = np.array([r.w_gt, r.d_gt, r.h_gt]) if r.mapped else None
    ax3 = fig.add_subplot(nrow, ncol, 2 * j + 2, projection='3d')
    cuboid(ax3, *pred, '#ef6c00', '-', f'예측 {fmt(pred)}')
    if gt is not None: cuboid(ax3, *gt, '#2e7d32', '--', f'정답 {fmt(gt)}')
    L = np.nanmax(np.r_[pred, gt if gt is not None else []]) * 1.05
    ax3.set_xlim(0, L); ax3.set_ylim(0, L); ax3.set_zlim(0, L); ax3.set_box_aspect((1, 1, 1)); ax3.view_init(elev=22, azim=-55)
    ax3.set_xlabel('w'); ax3.set_ylabel('d'); ax3.set_zlabel('h'); ax3.tick_params(labelsize=7); ax3.legend(fontsize=7, loc='upper left')
    ax3.set_title(f'T{tid} · 오차 {np.abs(sort3(pred) - sort3(gt)).mean():.1f} cm' if gt is not None else f'T{tid} · 정답 없음', fontsize=10)
cap.release(); plt.tight_layout(); plt.show()

Output hidden; open in https://colab.research.google.com to view.

# **11. 결과 정리** (ablation [2C-13] · 이 노트북 [E-1] · [T-1] · [R-4])

> 최종 모델: 가려짐 적은 프레임 최대 8장 → 고정 DA3 인코더 ⊕ DINOv3-L 7 × 7 토큰 → STEP 1 head (log B) · STEP 2 head (log d/w, log h/w) → 트랙마다 평균. 학습 파라미터 약 1.8 M. 모델 · 설정은 train · val로만 골랐고 **test는 한 번만 채점**했다.

## 11-1. 전체 성능 (수작업 매핑 쌍)
| | ① 가로 w | ② 치수 오차 | ③ + 벌점 | ② 납작 (≤ 10 cm) |
|---|---|---|---|---|
| train (5-fold OOF) | 0.84 cm | 1.80 cm · 9.2% | 2.04 cm | 12.3% |
| val | 1.25 cm | 1.95 cm · 10.3% | 2.44 cm | 15.4% |
| **test** | **1.02 cm · 4.3%** | **1.75 cm · 8.9%** [1.58, 1.93] | **2.16 cm · 11.7%** [1.81, 2.61] | **12.4%** |

- 같은 bbox · focal · 대표 프레임을 준 OVMono3D-LIFT: 원본 test 8.68 cm · 레일 보정 6.81 cm · 현장 데이터로 미세조정 3.45 cm ([T-2])
- 변별 오차 (test): 긴 변 2.19 cm · 7.0% · 중간 변 1.54 cm · 7.9% · 짧은 변 1.51 cm · 11.8%
- 검출 · 추적의 손해 (③ − ②, test): 0.40 cm

## 11-2. 오차 몫 분해 (test, 지표 ②)
| STEP 1만 (하한) | STEP 2만 | 둘 다 | STEP 1 몫 | corr(w, d/w) |
|---|---|---|---|---|
| 0.85 cm | 1.72 cm | 1.75 cm | 48% | −0.71 |

- 남은 오차의 절반 가까이가 가로 w이고, w 오차는 어떤 표현으로도 거의 같은 상자에서 같은 만큼 틀린다 → bbox · 레일 스케일 · w/d 분할의 모호함이 남은 과제

## 11-3. test 구간별 (지표 ②, 영상 단위 bootstrap 95% CI)
| 구간 | 상자 (영상) | cm | % |
|---|---|---|---|
| 가장 긴 변 ~ 30 cm | 219 (49) | 1.50 [1.34, 1.66] | 9.8 [8.8, 11.0] |
| 30 ~ 40 cm | 151 (46) | 1.97 [1.63, 2.31] | 8.8 [7.4, 10.3] |
| 40 ~ 50 cm | 66 (39) | 1.88 [1.59, 2.24] | 7.0 [5.9, 8.4] |
| 50 cm ~ | 33 (25) | 2.17 [1.70, 2.79] | 6.9 [5.7, 8.3] |
| 짧은 변 ≤ 10 cm (납작) | 110 (45) | 1.80 [1.42, 2.27] | 12.4 [10.4, 14.6] |
| 짧은 변 10 ~ 15 cm | 181 (50) | 1.76 [1.54, 1.99] | 9.0 [8.0, 10.1] |
| 짧은 변 > 15 cm | 178 (44) | 1.71 [1.50, 1.92] | 6.6 [5.7, 7.5] |
| 장면 2 ~ 5개 | 37 (9) | 1.54 [1.24, 1.90] | 8.6 [7.1, 10.4] |
| 장면 6 ~ 10개 | 167 (21) | 1.49 [1.31, 1.69] | 7.6 [6.7, 8.5] |
| 장면 11개 ~ | 265 (20) | 1.94 [1.69, 2.20] | 9.8 [8.7, 10.9] |

## 11-4. 런타임 ([R-0] 환경 · [R-4] 요약을 옮겨 적기)
| 처리 범위 · 방식 | 박스당 시간 | 실시간 배율 (가장 느린 영상) | 실시간 |
|---|---|---|---|
| 치수 추정만 · 기존 (seek 읽기 · 한 장씩) | [R-4] | [R-4] | |
| 치수 추정만 · 배치 (순차 읽기 · 영상 단위 배치) | [R-4] | [R-4] | |
| 치수 추정만 · 스트리밍 (트랙 단위 배치) | [R-4] | [R-4] | |
| **전체 파이프라인 · YOLO 한 장씩** | [R-4] | [R-4] | |
| **전체 파이프라인 · YOLO 16장씩** | [R-4] | [R-4] | |

- 배치 크기는 GPU가 포화되는 지점까지 키웠다 ([R-1]). torch.compile · TensorRT · GPU 디코딩은 하지 않았다
- 전체 파이프라인은 main.py 추적(영상 전체를 본 뒤 결과)이라 처리량 기준 실시간이다

## 11-5. 저장 파일
| 파일 | 내용 |
|---|---|
| `outputs/step2/result.json` | test 제출 형식 (영상 50개 · 트랙 477개) |
| `outputs/step2/{split}_proposed_preds.csv` | 트랙별 w · d · h |
| `outputs/step2/TEST_SCORED.txt` | test 채점 기록 |
| `outputs/step2/runtime_summary.json` | 런타임 환경 · 영상별 · 요약 |
| `outputs/step2/vis/test_XXX_result.mp4` | 시각화 영상 |
| `outputs/binpack/{split}_binpack_input.csv` | bin packing 입력 |